# IPL 2026 Analytics — Data Audit

## Project Objective

This project analyzes the IPL 2026 season using ball-by-ball cricket data.

The complete workflow will cover:

1. Data Audit
2. Data Cleaning
3. Exploratory Data Analysis
4. Power BI Dashboard
5. Documentation and Recruiter Story

## Data Strategy

The `deliveries.csv` file is treated as the primary source of truth
for season-level analysis because it contains ball-by-ball data for
the complete 2026 season.

The `matches.csv` file contains additional match-level information,
but it does not contain the complete season.

This notebook only audits the raw data.
No permanent cleaning or modification is performed here.

## 1. Import Required Libraries

We use:

- **Pandas** for loading and analyzing tabular data.
- **NumPy** for numerical operations.

Visualization libraries will be used later during the EDA stage.

In [3]:
import pandas as pd
import numpy as np

In [4]:
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Pandas version: 2.3.3
NumPy version: 2.3.5


## 2. Load Raw Datasets

The `Original-Data` folder contains the raw IPL 2026 datasets.

At this stage, we load all available files, including the Orange Cap
and Purple Cap tables.

We will not assume that any file is correct or incorrect based only
on its filename.

Instead, we will audit the contents and compare them with statistics
calculated from the ball-by-ball data.

This is important because different datasets may represent different
snapshots of the same IPL 2026 season.

In [5]:
import pandas as pd
import numpy as np

# Load main datasets
matches = pd.read_csv("../Original-Data/matches.csv")
deliveries = pd.read_csv("../Original-Data/deliveries.csv")

# Load supporting datasets
batting_stats = pd.read_csv("../Original-Data/batting_stats.csv")
bowling_stats = pd.read_csv("../Original-Data/bowling_stats.csv")
fielding_stats = pd.read_csv("../Original-Data/fielding_stats.csv")
points_table = pd.read_csv("../Original-Data/points_table.csv")

# Load cap datasets
orange_cap = pd.read_csv("../Original-Data/orange_cap.csv")
purple_cap = pd.read_csv("../Original-Data/purple_cap.csv")

# Load reference datasets
squads = pd.read_csv("../Original-Data/squads.csv")
venues = pd.read_csv("../Original-Data/venues.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


## 3. Dataset Inventory

We first inspect the size of every raw dataset.

The purpose is to understand what data is available before making
any assumptions about its completeness or accuracy.

In [7]:
datasets = {
    "matches": matches,
    "deliveries": deliveries,
    "batting_stats": batting_stats,
    "bowling_stats": bowling_stats,
    "fielding_stats": fielding_stats,
    "points_table": points_table,
    "orange_cap": orange_cap,
    "purple_cap": purple_cap,
    "squads": squads,
    "venues": venues
}

for name, df in datasets.items():
    print(f"{name:20} {df.shape}")

matches              (39, 23)
deliveries           (17477, 19)
batting_stats        (15, 17)
bowling_stats        (15, 17)
fielding_stats       (15, 6)
points_table         (10, 9)
orange_cap           (15, 16)
purple_cap           (15, 14)
squads               (257, 6)
venues               (12, 5)


## 4. Orange Cap and Purple Cap Audit

The Orange Cap and Purple Cap files contain player rankings and
statistics.

Before using them in the final analysis, we need to determine whether
they represent the same season snapshot as our complete
ball-by-ball dataset.

We will compare these tables with statistics calculated directly
from `deliveries.csv`.

The ball-by-ball data will be used as the primary source for
recalculating final season statistics.

In [8]:
print("Orange Cap:")
display(orange_cap.head())

print("\nPurple Cap:")
display(purple_cap.head())

Orange Cap:


,Position,Batsman,Team,Matches,Innings,Not_out,Runs,Highest_score,Average,Balls_faced,Strike_rate,Hundreds,Fifties,Ducks,Fours,Sixes
0,1,Sai Sudharsan,GT,15,15,1,759,108,54.21,486,156.17,1,6,0,88,21
1,2,Suryakumar Yadav,MI,16,16,5,717,73,65.18,427,167.91,0,5,0,69,38
2,3,Virat Kohli,RCB,15,15,3,657,73,54.75,454,144.71,0,8,0,66,19
3,4,Shubman Gill,GT,15,15,2,650,93,50.00,417,155.87,0,6,0,62,24
4,5,Mitchell Marsh,LSG,13,13,0,627,117,48.23,383,163.70,1,6,2,56,37



Purple Cap:


,Position,Bowler,Team,Matches,Innings,Balls,Overs,Maidens,Runs,Wickets,Best_bowling_figure,Economy_rate,Four_wicket_haul,Five_wicket_hall
0,1,Prasidh Krishna,GT,15,15,354,59.0,0,488,25,4--41,8.27,1,0
1,2,Noor Ahmad,CSK,14,14,300,50.0,0,408,24,4--18,8.16,2,0
2,3,Josh Hazlewood,RCB,12,12,264,44.0,0,386,22,4--33,8.77,1,0
3,4,Trent Boult,MI,16,16,346,57.4,0,517,22,4--26,8.96,1,0
4,5,Arshdeep Singh,PBKS,17,16,350,58.2,1,518,21,3--16,8.88,0,0


In [6]:
print("Orange Cap columns:")
print(orange_cap.columns.tolist())

print("\nPurple Cap columns:")
print(purple_cap.columns.tolist())

Orange Cap columns:
['Position', 'Batsman', 'Team', 'Matches', 'Innings', 'Not_out', 'Runs', 'Highest_score', 'Average', 'Balls_faced', 'Strike_rate', 'Hundreds', 'Fifties', 'Ducks', 'Fours', 'Sixes']

Purple Cap columns:
['Position', 'Bowler', 'Team', 'Matches', 'Innings', 'Balls', 'Overs', 'Maidens', 'Runs', 'Wickets', 'Best_bowling_figure', 'Economy_rate', 'Four_wicket_haul', 'Five_wicket_hall']


In [7]:
print("Orange Cap leader:")
print(
    orange_cap.loc[
        orange_cap["Position"].idxmin(),
        ["Batsman", "Team", "Matches", "Runs"]
    ]
)

print("\nPurple Cap leader:")
print(
    purple_cap.loc[
        purple_cap["Position"].idxmin(),
        ["Bowler", "Team", "Matches", "Wickets"]
    ]
)

Orange Cap leader:
Batsman    Sai Sudharsan
Team                  GT
Matches               15
Runs                 759
Name: 0, dtype: object

Purple Cap leader:
Bowler     Prasidh Krishna
Team                    GT
Matches                 15
Wickets                 25
Name: 0, dtype: object


## 5. Match-Level Data Audit

In this section, we will:
- Check match coverage in the deliveries dataset.
- Identify matches with missing or unexpected delivery records.
- Compare innings-level runs with match-level totals.
- Investigate unusual match records.

In [8]:
# Unique match IDs in both datasets
delivery_match_ids = set(deliveries["match_no"].dropna().unique())
match_table_ids = set(matches["match_id"].dropna().unique())

print("Matches in deliveries:", len(delivery_match_ids))
print("Matches in matches table:", len(match_table_ids))

print("\nMatches only in deliveries:")
print(sorted(delivery_match_ids - match_table_ids))

print("\nMatches only in matches table:")
print(sorted(match_table_ids - delivery_match_ids))

Matches in deliveries: 74
Matches in matches table: 39

Matches only in deliveries:
[np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(74)]

Matches only in matches table:
[]


### 5.2 Innings Coverage

Check the number of innings recorded for each match and identify
matches with an unusual innings count.

In [9]:
# Count distinct innings recorded for each match
innings_per_match = (
    deliveries.groupby("match_no")["innings"]
    .nunique()
    .reset_index(name="innings_count")
)

print("Innings count distribution:")
print(innings_per_match["innings_count"].value_counts().sort_index())

print("\nMatches with more than 2 innings:")
print(
    innings_per_match[innings_per_match["innings_count"] > 2]
    .sort_values("match_no")
    .to_string(index=False)
)

Innings count distribution:
innings_count
1     1
2    72
4     1
Name: count, dtype: int64

Matches with more than 2 innings:
 match_no  innings_count
       38              4


### 5.3 Investigating Unusual Innings

Inspect matches with fewer or more than two innings.
Calculate total runs from batting runs and extras,
and check delivery counts for unusual innings.

In [11]:
# Identify matches with unusual innings counts
unusual_matches = innings_per_match[
    innings_per_match["innings_count"] != 2
]["match_no"].tolist()

# Calculate total runs and summarize each innings
unusual_innings = (
    deliveries[deliveries["match_no"].isin(unusual_matches)]
    .groupby(["match_no", "innings"])
    .agg(
        delivery_rows=("innings", "size"),
        batting_runs=("runs_of_bat", "sum"),
        extras=("extras", "sum")
    )
    .reset_index()
)

# Calculate total runs
unusual_innings["total_runs"] = (
    unusual_innings["batting_runs"] + unusual_innings["extras"]
)

unusual_innings = unusual_innings.sort_values(
    ["match_no", "innings"]
)

print(unusual_innings.to_string(index=False))

 match_no  innings  delivery_rows  batting_runs  extras  total_runs
       12        1             22            25       0          25
       38        1            126           147       7         154
       38        2            125           147       8         155
       38        3              3             1       0           1
       38        4              1             4       0           4


### 5.4 Inspecting Unusual Match Records

Inspect the delivery-level records of matches with unusual
innings counts to identify possible abandoned matches,
Super Overs, or incomplete data.

In [12]:
# Inspect all delivery records for Match 38
match_38 = deliveries[deliveries["match_no"] == 38]

print("Match 38 innings summary:")
print(
    match_38.groupby("innings")
    .agg(
        delivery_rows=("innings", "size"),
        batting_team=("batting_team", "first"),
        bowling_team=("bowling_team", "first"),
        total_runs=("runs_of_bat", "sum")
    )
)

print("\nMatch 38 additional innings (3 and 4):")
print(
    match_38[match_38["innings"].isin([3, 4])].to_string(index=False)
)

print("\nMatch 12 delivery records:")
print(
    deliveries[deliveries["match_no"] == 12].to_string(index=False)
)

Match 38 innings summary:
         delivery_rows batting_team bowling_team  total_runs
innings                                                     
1                  126          KKR          LSG         147
2                  125          LSG          KKR         147
3                    3          LSG          KKR           1
4                    1          KKR          LSG           4

Match 38 additional innings (3 and 4):
 match_no         date        stage                                                                 venue batting_team bowling_team  innings  over         striker       bowler  runs_of_bat  extras  wide  legbyes  byes  noballs wicket_type player_dismissed     fielder
       38 Apr 26, 2026 League stage Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow          LSG          KKR        3   0.1 Nicholas Pooran Sunil Narine            0       0     0        0     0        0      bowled  Nicholas Pooran         NaN
       38 Apr 26, 2026 League st

### 5.5 Match-Level Audit Findings

- The deliveries dataset contains records for 74 matches, while
  the matches dataset contains details for only 39.
- Match 12 has one partial innings with 22 delivery rows and
  25 runs recorded. Its official status requires verification.
- Match 38 contains four innings. Innings 3 and 4 appear to
  represent a Super Over and should be excluded from regular
  innings statistics.
- Raw delivery records will be preserved. Any exclusions will
  be applied through documented analysis filters rather than
  deleting source records.

### 5.6 Comparing Delivery Totals with Match Records

Compare the total runs calculated from delivery-level data
with the available match-level score information.

This check will help identify discrepancies in recorded
innings totals.

In [13]:
print("Columns in matches dataset:")
print(matches.columns.tolist())

print("\nSample match records:")
display(matches.head())

Columns in matches dataset:
['match_id', 'date', 'venue', 'team1', 'team2', 'stage', 'toss_winner', 'toss_decision', 'first_ings_score', 'first_ings_wkts', 'second_ings_score', 'second_ings_wkts', 'match_result', 'match_winner', 'wb_runs', 'wb_wickets', 'balls_left', 'player_of_the_match', 'top_scorer', 'highscore', 'best_bowling', 'best_bowling_figure', 'super_over_match']

Sample match records:


,match_id,date,venue,team1,team2,stage,toss_winner,toss_decision,first_ings_score,first_ings_wkts,...,match_winner,wb_runs,wb_wickets,balls_left,player_of_the_match,top_scorer,highscore,best_bowling,best_bowling_figure,super_over_match
0,1,"March 28, 2026","M. Chinnaswamy Stadium, Bangalore",RCB,SRH,League,RCB,Bowl,201.0,9.0,...,RCB,NaN,6.0,26.0,Jacob Duffy,Ishan Kishan,80.0,Jacob Duffy,3--22,No
1,2,"March 29, 2026","Wankhede Stadium, Mumbai",MI,KKR,League,MI,Bowl,220.0,4.0,...,MI,NaN,6.0,5.0,Shardul Thakur,Ryan Rickelton,81.0,Shardul Thakur,3--39,No
2,3,"March 30, 2026","Barsapara Stadium, Guwahati",RR,CSK,League,RR,Bowl,127.0,10.0,...,RR,NaN,8.0,47.0,Nandre Burger,Vaibhav Sooryavanshi,52.0,Jofra Archer,2--19,No
3,4,"March 31, 2026","New PCA Cricket Stadium, Mullanpur",PBKS,GT,League,PBKS,Bowl,162.0,6.0,...,PBKS,NaN,3.0,5.0,Cooper Connolly,Cooper Connolly,72.0,Prasidh Krishna,3--29,No
4,5,"April 01, 2026","Ekana Cricket Stadium, Lucknow",LSG,DC,League,DC,Bowl,141.0,10.0,...,DC,NaN,6.0,17.0,Sameer Rizvi,Sameer Rizvi,70.0,Lungi Ngidi,3--27,No


### 5.7 Comparing Innings Scores

Calculate the total runs scored in the first and second
innings from the deliveries dataset.

Compare these totals with the corresponding scores in
the matches dataset and identify any mismatches.

In [14]:
# Calculate total runs from deliveries
deliveries["calculated_runs"] = (
    deliveries["runs_of_bat"] + deliveries["extras"]
)

# Exclude Super Over innings from regular innings totals
regular_innings = deliveries[deliveries["innings"].isin([1, 2])]

# Calculate first and second innings totals
innings_totals = (
    regular_innings
    .groupby(["match_no", "innings"])["calculated_runs"]
    .sum()
    .unstack()
    .rename(columns={
        1: "calculated_first_innings",
        2: "calculated_second_innings"
    })
    .reset_index()
)

# Compare with the match-level data
score_comparison = matches.merge(
    innings_totals,
    left_on="match_id",
    right_on="match_no",
    how="left"
)

# Display relevant columns
score_comparison = score_comparison[
    [
        "match_id",
        "first_ings_score",
        "calculated_first_innings",
        "second_ings_score",
        "calculated_second_innings"
    ]
]

display(score_comparison.head(10))

,match_id,first_ings_score,calculated_first_innings,second_ings_score,calculated_second_innings
0,1,201.0,203.0,203.0,203.0
1,2,220.0,220.0,224.0,224.0
2,3,127.0,128.0,128.0,128.0
3,4,162.0,162.0,165.0,165.0
4,5,141.0,141.0,145.0,144.0
5,6,226.0,226.0,161.0,160.0
6,7,209.0,209.0,210.0,209.0
7,8,162.0,162.0,164.0,164.0
8,9,210.0,210.0,204.0,204.0
9,10,156.0,156.0,160.0,160.0


### 5.8 Identifying Score Mismatches

Calculate the difference between match-table scores and
delivery-level totals for both innings.

Identify all matches where the scores do not match.

In [15]:
# Calculate score differences
score_comparison["first_innings_difference"] = (
    score_comparison["calculated_first_innings"]
    - score_comparison["first_ings_score"]
)

score_comparison["second_innings_difference"] = (
    score_comparison["calculated_second_innings"]
    - score_comparison["second_ings_score"]
)

# Filter mismatched scores
score_mismatches = score_comparison[
    (score_comparison["first_innings_difference"] != 0)
    | (score_comparison["second_innings_difference"] != 0)
]

print("Total matches compared:", len(score_comparison))
print("Matches with score mismatches:", len(score_mismatches))

display(score_mismatches)

Total matches compared: 39
Matches with score mismatches: 20


,match_id,first_ings_score,calculated_first_innings,second_ings_score,calculated_second_innings,first_innings_difference,second_innings_difference
0,1,201.0,203.0,203.0,203.0,2.0,0.0
2,3,127.0,128.0,128.0,128.0,1.0,0.0
4,5,141.0,141.0,145.0,144.0,0.0,-1.0
5,6,226.0,226.0,161.0,160.0,0.0,-1.0
6,7,209.0,209.0,210.0,209.0,0.0,-1.0
11,12,NaN,25.0,NaN,NaN,NaN,NaN
13,14,210.0,210.0,209.0,208.0,0.0,-1.0
16,17,219.0,216.0,223.0,223.0,-3.0,0.0
19,20,240.0,239.0,222.0,222.0,-1.0,0.0
20,21,216.0,214.0,159.0,159.0,-2.0,0.0


### 5.9 Investigating Run Differences

Break down batting runs and extras for innings with mismatched
scores. Compare the recorded extras with the individual
extras categories to identify possible inconsistencies.

In [16]:
# Get match IDs with score mismatches, excluding missing scores
mismatch_ids = score_mismatches["match_id"].dropna().astype(int).tolist()

# Keep only regular innings from mismatched matches
mismatch_deliveries = deliveries[
    deliveries["match_no"].isin(mismatch_ids)
    & deliveries["innings"].isin([1, 2])
].copy()

# Summarize runs and extras by match and innings
extras_audit = (
    mismatch_deliveries
    .groupby(["match_no", "innings"])
    .agg(
        delivery_rows=("innings", "size"),
        batting_runs=("runs_of_bat", "sum"),
        recorded_extras=("extras", "sum"),
        wides=("wide", "sum"),
        legbyes=("legbyes", "sum"),
        byes=("byes", "sum"),
        noballs=("noballs", "sum")
    )
    .reset_index()
)

extras_audit["calculated_total"] = (
    extras_audit["batting_runs"] + extras_audit["recorded_extras"]
)

display(extras_audit.head(15))

,match_no,innings,delivery_rows,batting_runs,recorded_extras,wides,legbyes,byes,noballs,calculated_total
0,1,1,126,196,7,5,2,0,0,203
1,1,2,102,185,18,7,2,3,0,203
2,3,1,125,112,16,6,6,0,1,128
3,3,2,74,122,6,1,1,1,0,128
4,5,1,114,127,14,2,2,1,0,141
5,5,2,114,125,19,11,3,0,0,144
6,6,1,129,213,13,9,1,0,0,226
7,6,2,101,153,7,5,1,0,0,160
8,7,1,131,192,17,10,0,1,1,209
9,7,2,119,201,8,5,1,0,2,209


# Verify the audit table

In [17]:
print("Actual columns:")
print(extras_audit.columns.tolist())

print("\nFirst 5 rows:")
print(extras_audit.head().to_string(index=False))

print("\nCheck calculated totals:")
print(
    extras_audit[
        ["match_no", "innings", "batting_runs",
         "recorded_extras", "calculated_total"]
    ].head().assign(
        verified_total=lambda df:
            df["batting_runs"] + df["recorded_extras"]
    ).to_string(index=False)
)

Actual columns:
['match_no', 'innings', 'delivery_rows', 'batting_runs', 'recorded_extras', 'wides', 'legbyes', 'byes', 'noballs', 'calculated_total']

First 5 rows:
 match_no  innings  delivery_rows  batting_runs  recorded_extras  wides  legbyes  byes  noballs  calculated_total
        1        1            126           196                7      5        2     0        0               203
        1        2            102           185               18      7        2     3        0               203
        3        1            125           112               16      6        6     0        1               128
        3        2             74           122                6      1        1     1        0               128
        5        1            114           127               14      2        2     1        0               141

Check calculated totals:
 match_no  innings  batting_runs  recorded_extras  calculated_total  verified_total
        1        1           196        

### 5.10 Inspecting a Score Mismatch

Inspect the extras recorded in Match 1, first innings.
Compare the individual extras categories with the total
extras to investigate the 2-run discrepancy.

In [18]:
# Filter Match 1, first innings
match1_innings1 = deliveries[
    (deliveries["match_no"] == 1) &
    (deliveries["innings"] == 1)
].copy()

# Show rows containing extras
match1_extras = match1_innings1[
    match1_innings1["extras"] > 0
]

print("Extras-related deliveries:")
display(
    match1_extras[
        [
            "over", "striker", "bowler", "runs_of_bat",
            "extras", "wide", "legbyes", "byes", "noballs"
        ]
    ]
)

print("\nExtras totals:")
print(
    match1_innings1[
        ["extras", "wide", "legbyes", "byes", "noballs"]
    ].sum()
)

Extras-related deliveries:


,over,striker,bowler,runs_of_bat,extras,wide,legbyes,byes,noballs
10,1.4,Abhishek Sharma,Bhuvneshwar Kumar,0,1,1,0,0,0
11,1.5,Abhishek Sharma,Bhuvneshwar Kumar,0,1,0,1,0,0
12,1.5,Abhishek Sharma,Bhuvneshwar Kumar,0,1,1,0,0,0
15,2.1,Abhishek Sharma,Jacob Duffy,0,1,1,0,0,0
16,2.1,Abhishek Sharma,Jacob Duffy,0,1,1,0,0,0
105,16.5,Harsh Dubey,Romario Shepherd,0,1,1,0,0,0
122,19.3,David Payne,Bhuvneshwar Kumar,0,1,0,1,0,0



Extras totals:
extras     7
wide       5
legbyes    2
byes       0
noballs    0
dtype: int64


### 5.11 Checking Repeated Over-Ball Labels

Identify repeated over-ball labels in Match 1, first innings.
Inspect whether these rows represent extra deliveries or
potential duplicate records.

In [19]:
# Find repeated over-ball labels
repeated_balls = match1_innings1[
    match1_innings1.duplicated(subset=["over"], keep=False)
].sort_values("over")

print("Repeated over-ball labels:")
display(
    repeated_balls[
        [
            "over", "striker", "bowler", "runs_of_bat",
            "extras", "wide", "legbyes", "byes", "noballs"
        ]
    ]
)

Repeated over-ball labels:


,over,striker,bowler,runs_of_bat,extras,wide,legbyes,byes,noballs
9,1.4,Abhishek Sharma,Bhuvneshwar Kumar,0,0,0,0,0,0
10,1.4,Abhishek Sharma,Bhuvneshwar Kumar,0,1,1,0,0,0
11,1.5,Abhishek Sharma,Bhuvneshwar Kumar,0,1,0,1,0,0
12,1.5,Abhishek Sharma,Bhuvneshwar Kumar,0,1,1,0,0,0
14,2.1,Abhishek Sharma,Jacob Duffy,0,0,0,0,0,0
15,2.1,Abhishek Sharma,Jacob Duffy,0,1,1,0,0,0
16,2.1,Abhishek Sharma,Jacob Duffy,0,1,1,0,0,0
66,10.3,Ishan Kishan,Krunal Pandya,1,0,0,0,0,0
67,10.3,Ishan Kishan,Krunal Pandya,1,0,0,0,0,0
105,16.5,Harsh Dubey,Romario Shepherd,0,1,1,0,0,0


### 5.12 Inspecting Repeated Delivery Sequences

Inspect the delivery sequence around repeated over-ball labels.
Check whether repeated records represent legitimate deliveries
or possible duplicate entries.

In [20]:
# Inspect the surrounding delivery sequence
suspicious_rows = [9, 10, 11, 12, 14, 15, 16, 66, 67, 105, 106]

start = max(min(suspicious_rows) - 2, 0)
end = max(suspicious_rows) + 3

print("Delivery sequence around suspicious rows:")

display(
    match1_innings1.iloc[start:end][
        [
            "over", "striker", "bowler", "runs_of_bat",
            "extras", "wide", "legbyes", "byes", "noballs"
        ]
    ]
)

Delivery sequence around suspicious rows:


,over,striker,bowler,runs_of_bat,extras,wide,legbyes,byes,noballs
7,1.2,Abhishek Sharma,Bhuvneshwar Kumar,1,0,0,0,0,0
8,1.3,Travis Head,Bhuvneshwar Kumar,1,0,0,0,0,0
9,1.4,Abhishek Sharma,Bhuvneshwar Kumar,0,0,0,0,0,0
10,1.4,Abhishek Sharma,Bhuvneshwar Kumar,0,1,1,0,0,0
11,1.5,Abhishek Sharma,Bhuvneshwar Kumar,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...
104,16.4,Aniket Verma,Romario Shepherd,1,0,0,0,0,0
105,16.5,Harsh Dubey,Romario Shepherd,0,1,1,0,0,0
106,16.5,Harsh Dubey,Romario Shepherd,2,0,0,0,0,0
107,16.6,Harsh Dubey,Romario Shepherd,0,0,0,0,0,0


### 5.13 Checking Exact Duplicate Records

Identify rows that are identical across all columns
in the deliveries dataset.

Repeated over-ball labels are not necessarily duplicates,
so this check compares complete records.

In [21]:
# Count exact duplicate rows across all columns
duplicate_count = deliveries.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

# Display exact duplicate records, if any
exact_duplicates = deliveries[
    deliveries.duplicated(keep=False)
].sort_values(["match_no", "innings", "over"])

display(exact_duplicates.head(20))

Exact duplicate rows: 0


,match_no,date,stage,venue,batting_team,bowling_team,innings,over,striker,bowler,runs_of_bat,extras,wide,legbyes,byes,noballs,wicket_type,player_dismissed,fielder,calculated_runs


### Exact Duplicate Records

The deliveries dataset was checked for rows that are identical across all columns.

- Exact duplicate rows: 0
- No exact duplicate records were found.
- Repeated over-ball labels were not treated as duplicates because they can represent separate deliveries or events.

**Conclusion:** No exact duplicate rows need to be removed based on this check.

In [22]:
# Check missing values in each column
missing_values = deliveries.isnull().sum()

missing_summary = (
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(deliveries) * 100
).round(2)

display(missing_summary)

,missing_count,missing_percentage
fielder,16791,96.07
player_dismissed,16602,94.99
wicket_type,16598,94.97


# Validate wicket-related missing values 

In [23]:
# Count rows with a dismissal and rows with a wicket type
print("Rows with player dismissed:", deliveries["player_dismissed"].notna().sum())
print("Rows with wicket type:", deliveries["wicket_type"].notna().sum())
print("Rows with fielder:", deliveries["fielder"].notna().sum())

# Check whether dismissal and wicket type are present together
dismissal_check = pd.crosstab(
    deliveries["player_dismissed"].notna(),
    deliveries["wicket_type"].notna(),
    rownames=["Player dismissed present"],
    colnames=["Wicket type present"]
)

display(dismissal_check)

Rows with player dismissed: 875
Rows with wicket type: 879
Rows with fielder: 686


Wicket type present,False,True
Player dismissed present,,
False,16598,4
True,0,875


# Inspect the four unusual wicket records

In [24]:
# Rows with a wicket type but no dismissed player
unusual_wickets = deliveries[
    deliveries["wicket_type"].notna()
    & deliveries["player_dismissed"].isna()
]

display(
    unusual_wickets[
        [
            "match_no",
            "innings",
            "over",
            "striker",
            "bowler",
            "runs_of_bat",
            "extras",
            "wicket_type",
            "player_dismissed",
            "fielder"
        ]
    ]
)

,match_no,innings,over,striker,bowler,runs_of_bat,extras,wicket_type,player_dismissed,fielder
15,1,1,2.1,Abhishek Sharma,Jacob Duffy,0,1,bowled,NaN,NaN
66,1,1,10.3,Ishan Kishan,Krunal Pandya,1,0,bowled,NaN,NaN
169,1,2,7.2,Virat Kohli,Harshal Patel,0,0,caught,NaN,NaN
1637,8,1,0.6,Ryan Rickelton,Mukesh Kumar,0,0,caught,NaN,NaN


# Display the four records clearly

In [25]:
# Display the four unusual wicket records without truncation
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(unusual_wickets.to_string(index=False))

' match_no         date        stage                            venue batting_team bowling_team  innings  over         striker        bowler  runs_of_bat  extras  wide  legbyes  byes  noballs wicket_type player_dismissed fielder  calculated_runs\n        1 Mar 28, 2026 League stage M.Chinnaswamy Stadium, Bengaluru          SRH          RCB        1   2.1 Abhishek Sharma   Jacob Duffy            0       1     1        0     0        0      bowled              NaN     NaN                1\n        1 Mar 28, 2026 League stage M.Chinnaswamy Stadium, Bengaluru          SRH          RCB        1  10.3    Ishan Kishan Krunal Pandya            1       0     0        0     0        0      bowled              NaN     NaN                1\n        1 Mar 28, 2026 League stage M.Chinnaswamy Stadium, Bengaluru          RCB          SRH        2   7.2     Virat Kohli Harshal Patel            0       0     0        0     0        0      caught              NaN     NaN                0\n        8  Apr 

### Wicket Record Consistency Check

The wicket-related columns were checked for consistency.

- 875 rows contain both `player_dismissed` and `wicket_type`.
- 16,598 rows have both fields missing.
- 4 rows have a `wicket_type` but no `player_dismissed`.
- No rows have a `player_dismissed` value while `wicket_type` is missing.

The four unusual records include two `bowled` and two `caught` wicket types. Their missing dismissed-player values require further verification.

**Decision:** Preserve these records in the raw dataset. Do not impute the missing player names or remove the rows without additional evidence.

## Check wicket types

In [26]:
# Count wicket types
wicket_type_counts = (
    deliveries["wicket_type"]
    .value_counts(dropna=False)
    .rename_axis("wicket_type")
    .reset_index(name="count")
)

display(wicket_type_counts)

,wicket_type,count
0,NaN,16598
1,caught,660
2,bowled,137
3,runout,38
4,lbw,30
5,stumped,9
6,hit wicket,2
7,retired hurt,1
8,obstructing the field,1
9,hitwicket,1


### Wicket Type Distribution

The delivery dataset contains the following non-missing wicket labels:

- Caught: 660
- Bowled: 137
- Run out: 38
- LBW: 30
- Stumped: 9
- Hit wicket: 2
- Retired hurt: 1
- Obstructing the field: 1
- Hitwicket: 1

**Data-quality observations:**
- `hit wicket` and `hitwicket` appear to be spelling variants and should be standardized after verification.
- The `retired hurt` record requires separate review.
- Four rows have a wicket type but no dismissed-player value.

**Decision:** Preserve the original labels during the audit. Standardize categories only in the cleaned dataset, with the transformation documented.

## Inspect the unusual dismissal labels

In [27]:
# Inspect less common dismissal types
unusual_types = [
    "hit wicket",
    "hitwicket",
    "retired hurt",
    "obstructing the field"
]

display(
    deliveries[
        deliveries["wicket_type"].isin(unusual_types)
    ][
        [
            "match_no",
            "innings",
            "over",
            "striker",
            "bowler",
            "wicket_type",
            "player_dismissed",
            "fielder"
        ]
    ].sort_values(["match_no", "innings", "over"])
)

,match_no,innings,over,striker,bowler,wicket_type,player_dismissed,fielder
4416,20,2,5.2,Rohit Sharma,Rasikh Salam Dar,retired hurt,Rohit Sharma,NaN
8682,38,1,4.6,Angkrish Raghuvanshi,Prince Yadav,obstructing the field,Angkrish Raghuvanshi,NaN
16642,71,2,2.3,Sai Sudharsan,Jacob Duffy,hitwicket,Sai Sudharsan,NaN
16643,71,2,2.3,Sai Sudharsan,Jacob Duffy,hit wicket,Sai Sudharsan,NaN
17208,73,2,12.5,Sai Sudharsan,Brijesh Sharma,hit wicket,Sai Sudharsan,NaN


### Review of Uncommon Wicket Types

The less frequent dismissal labels were inspected.

- `retired hurt`: Rohit Sharma, match 20.
- `obstructing the field`: Angkrish Raghuvanshi, match 38.
- `hitwicket` and `hit wicket`: Both spellings occur in match 71.
- Two match 71 records share the same batter, bowler, dismissal type, and over label.
- All five inspected records have a missing fielder value.

**Data-quality decisions:**
- Flag `hitwicket` and `hit wicket` as possible spelling variants for standardization.
- Investigate the two match 71 records before treating them as duplicates.
- Preserve uncommon dismissal labels and missing fielder values until their meaning is verified.

## Compare the two match 71 records

In [28]:
# Inspect the two match 71 records and nearby deliveries
match71_innings2 = deliveries[
    (deliveries["match_no"] == 71) &
    (deliveries["innings"] == 2)
].reset_index()

display(
    match71_innings2[
        match71_innings2["over"].between(2.1, 2.6)
    ].to_string(index=False)
)

' index  match_no         date    stage                                                    venue batting_team bowling_team  innings  over       striker      bowler  runs_of_bat  extras  wide  legbyes  byes  noballs wicket_type player_dismissed fielder  calculated_runs\n 16640        71 May 26, 2026 Playoffs Himachal Pradesh Cricket Association Stadium, Dharamsala           GT          RCB        2   2.1 Sai Sudharsan Jacob Duffy            0       0     0        0     0        0         NaN              NaN     NaN                0\n 16641        71 May 26, 2026 Playoffs Himachal Pradesh Cricket Association Stadium, Dharamsala           GT          RCB        2   2.2 Sai Sudharsan Jacob Duffy            4       0     0        0     0        0         NaN              NaN     NaN                4\n 16642        71 May 26, 2026 Playoffs Himachal Pradesh Cricket Association Stadium, Dharamsala           GT          RCB        2   2.3 Sai Sudharsan Jacob Duffy            0       0     0   

## Check whether the spelling variant causes duplicate dismissals

In [30]:
# Normalize wicket-type spelling for comparison
wicket_check = deliveries.copy()

wicket_check["wicket_type_normalized"] = (
    wicket_check["wicket_type"]
    .replace({"hitwicket": "hit wicket"})
)

# Keep only rows with a dismissed player
dismissal_rows = wicket_check[
    wicket_check["player_dismissed"].notna()
].copy()

# Find repeated dismissal records after normalization
duplicate_mask = dismissal_rows.duplicated(
    subset=[
        "match_no",
        "innings",
        "over",
        "player_dismissed",
        "wicket_type_normalized"
    ],
    keep=False
)

# Display the matching records
display(
    dismissal_rows.loc[
        duplicate_mask,
        [
            "match_no",
            "innings",
            "over",
            "striker",
            "bowler",
            "wicket_type",
            "player_dismissed",
            "runs_of_bat",
            "extras"
        ]
    ].sort_values(["match_no", "innings", "over"])
)

,match_no,innings,over,striker,bowler,wicket_type,player_dismissed,runs_of_bat,extras
16642,71,2,2.3,Sai Sudharsan,Jacob Duffy,hitwicket,Sai Sudharsan,0,0
16643,71,2,2.3,Sai Sudharsan,Jacob Duffy,hit wicket,Sai Sudharsan,0,0


### Repeated Dismissal Check

After normalizing `hitwicket` to `hit wicket`, one repeated dismissal record was identified.

Both rows refer to:
- Match: 71
- Innings: 2
- Over: 2.3
- Batter dismissed: Sai Sudharsan
- Bowler: Jacob Duffy
- Dismissal type: Hit wicket

The two records are identical across the inspected delivery fields after spelling normalization.

**Decision:** Flag this pair as a likely duplicate dismissal. Preserve the raw data and verify the source before removing one row from the cleaned dataset.

## Check missing values in essential delivery columns

### Missing Values in Essential Delivery Columns

This check identifies missing values in the core columns required for delivery-level analysis, including match, innings, batter, bowler, and run information.

The results will help determine whether any records need further investigation before cleaning.

In [31]:
# Essential columns for delivery-level analysis
essential_columns = [
    "match_no",
    "innings",
    "over",
    "batting_team",
    "bowling_team",
    "striker",
    "bowler",
    "runs_of_bat",
    "extras"
]

# Count missing values and calculate percentages
essential_missing = deliveries[essential_columns].isnull().sum()

essential_missing_summary = pd.DataFrame({
    "missing_count": essential_missing,
    "missing_percentage": (
        essential_missing / len(deliveries) * 100
    ).round(2)
})

display(essential_missing_summary)

,missing_count,missing_percentage
match_no,0,0.0
innings,0,0.0
over,0,0.0
batting_team,0,0.0
bowling_team,0,0.0
striker,0,0.0
bowler,0,0.0
runs_of_bat,0,0.0
extras,0,0.0


### Missing Values in Essential Delivery Columns

The essential delivery columns were checked for missing values.

- All 9 selected columns have zero missing values.
- Match identifiers, innings, over labels, team names, batter, bowler, runs, and extras are complete.
- No missing-value treatment is required for these columns based on this audit.

**Conclusion:** The essential delivery fields are complete. Wicket-related missing values are handled separately because some are expected for deliveries without dismissals.

## Check data types

In [32]:
# Inspect data types of all delivery columns
display(
    pd.DataFrame({
        "column": deliveries.columns,
        "data_type": deliveries.dtypes.astype(str).values
    })
)

,column,data_type
0,match_no,int64
1,date,object
2,stage,object
3,venue,object
4,batting_team,object
5,bowling_team,object
6,innings,int64
7,over,float64
8,striker,object
9,bowler,object


### Data Type Audit

The data types of all delivery columns were inspected.

- Integer types are used for match identifiers, innings, and run-related columns.
- Date values are currently stored as text and may be converted to datetime during cleaning.
- Team, player, venue, stage, and wicket-type columns are stored as text.
- The `over` column is stored as a float. It represents an over-ball label, so it should not be treated as a conventional decimal number.

**Conclusion:** Most data types are suitable. Date conversion and careful handling of the over-ball notation are potential cleaning tasks.

## Inspect the date format and validity

In [33]:
# Inspect date values
print("Sample date values:")
display(deliveries["date"].head(10))

# Check date parsing
parsed_dates = pd.to_datetime(
    deliveries["date"],
    errors="coerce"
)

print("Unparseable dates:", parsed_dates.isna().sum())
print("Earliest date:", parsed_dates.min())
print("Latest date:", parsed_dates.max())

Sample date values:


0    Mar 28, 2026
1    Mar 28, 2026
2    Mar 28, 2026
3    Mar 28, 2026
4    Mar 28, 2026
5    Mar 28, 2026
6    Mar 28, 2026
7    Mar 28, 2026
8    Mar 28, 2026
9    Mar 28, 2026
Name: date, dtype: object

Unparseable dates: 0
Earliest date: 2026-03-28 00:00:00
Latest date: 2026-05-31 00:00:00


### Date Format and Validity Check

The `date` column was checked for consistency and successful datetime parsing.

- All dates were successfully parsed.
- Unparseable dates: 0
- Earliest date: March 28, 2026
- Latest date: May 31, 2026

**Conclusion:** The date values are valid and can be converted to datetime format during the cleaning stage.

## Check the over-ball notation

In [34]:
# Inspect unique over-ball labels
over_values = sorted(deliveries["over"].unique())

print("Number of unique over-ball labels:", len(over_values))
print("First 30 labels:", over_values[:30])
print("Last 20 labels:", over_values[-20:])

Number of unique over-ball labels: 120
First 30 labels: [np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(1.6), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(2.5), np.float64(2.6), np.float64(3.1), np.float64(3.2), np.float64(3.3), np.float64(3.4), np.float64(3.5), np.float64(3.6), np.float64(4.1), np.float64(4.2), np.float64(4.3), np.float64(4.4), np.float64(4.5), np.float64(4.6)]
Last 20 labels: [np.float64(16.5), np.float64(16.6), np.float64(17.1), np.float64(17.2), np.float64(17.3), np.float64(17.4), np.float64(17.5), np.float64(17.6), np.float64(18.1), np.float64(18.2), np.float64(18.3), np.float64(18.4), np.float64(18.5), np.float64(18.6), np.float64(19.1), np.float64(19.2), np.float64(19.3), np.float64(19.4), np.float64(19.5), np.float64(19.6)]


### Over-Ball Notation Audit

The `over` column was inspected to understand its stored format.

- Number of unique labels: 120
- Labels range from `0.1` to `19.6`.
- The values follow an over.ball notation, where the digit after the decimal point represents the ball number.
- The column should not be treated as a conventional decimal measure.

**Conclusion:** The notation appears consistent with standard 20-over cricket innings. Further sequence-level validation would be required to identify missing or unusual delivery labels.

## Check the over-ball label range

In [35]:
# Split over-ball labels into over number and ball number
over_text = deliveries["over"].map(lambda x: f"{x:.1f}")
parts = over_text.str.split(".", expand=True)

over_number = parts[0].astype(int)
ball_number = parts[1].astype(int)

# Identify labels outside the expected range
invalid_labels = deliveries[
    (over_number < 0) |
    (over_number > 19) |
    (ball_number < 1) |
    (ball_number > 6)
]

print("Labels outside the expected range:", len(invalid_labels))
display(invalid_labels[["match_no", "innings", "over"]].head(20))

Labels outside the expected range: 0


,match_no,innings,over


### Over-Ball Label Range Validation

The over-ball labels were checked against the expected ranges.

- Invalid labels: 0
- Over numbers fall within 0–19.
- Ball numbers fall within 1–6.

**Conclusion:** No labels outside the expected range were found. The `over` column can be retained as a delivery label, with appropriate care during analysis.

## Check the delivery-level audit summary

In [36]:
print("DELIVERY DATA AUDIT SUMMARY")
print("-" * 35)
print("Total delivery rows:", len(deliveries))
print("Exact duplicate rows:", deliveries.duplicated().sum())
print("Missing match numbers:", deliveries["match_no"].isna().sum())
print("Missing innings values:", deliveries["innings"].isna().sum())
print("Missing batter names:", deliveries["striker"].isna().sum())
print("Missing bowler names:", deliveries["bowler"].isna().sum())
print("Missing runs of bat:", deliveries["runs_of_bat"].isna().sum())
print("Missing extras:", deliveries["extras"].isna().sum())
print("Invalid over-ball labels:", len(invalid_labels))

DELIVERY DATA AUDIT SUMMARY
-----------------------------------
Total delivery rows: 17477
Exact duplicate rows: 0
Missing match numbers: 0
Missing innings values: 0
Missing batter names: 0
Missing bowler names: 0
Missing runs of bat: 0
Missing extras: 0
Invalid over-ball labels: 0


## Delivery audit conclusion

### Delivery-Level Data Audit — Conclusion

The delivery dataset contains 17,477 rows. The audit found no exact duplicate rows, no missing values in the essential delivery columns, and no over-ball labels outside the expected range.

A few issues require attention during cleaning and analysis:
- Four records have a wicket type but no dismissed-player value.
- One likely duplicate dismissal was identified after standardizing wicket-type spelling.
- Some match scores differ from the match metadata.
- Match 12 contains partial delivery data.
- Match 38 includes additional innings consistent with a Super Over.

The original data will be preserved. Any corrections, exclusions, or transformations will be documented and applied only when justified.

## Section 6: Match Dataset Audit

This section evaluates the completeness and consistency of the match-level dataset.

The audit begins by checking missing values and duplicate match identifiers to identify potential issues before analysis.

In [37]:
# Check missing values
matches_missing = matches.isnull().sum()

missing_summary = pd.DataFrame({
    "missing_count": matches_missing,
    "missing_percentage": (
        matches_missing / len(matches) * 100
    ).round(2)
})

display(missing_summary[missing_summary["missing_count"] > 0])

# Check duplicate match IDs
print("Total match records:", len(matches))
print("Unique match IDs:", matches["match_id"].nunique())
print("Duplicate match IDs:", matches["match_id"].duplicated().sum())

,missing_count,missing_percentage
first_ings_score,1,2.56
first_ings_wkts,1,2.56
second_ings_score,1,2.56
second_ings_wkts,1,2.56
match_winner,1,2.56
wb_runs,24,61.54
wb_wickets,17,43.59
balls_left,1,2.56
player_of_the_match,1,2.56
top_scorer,1,2.56


Total match records: 39
Unique match IDs: 39
Duplicate match IDs: 0


## Step 6.2 — Inspect the match record with missing values

In [38]:
# Find match records with missing values
matches_with_missing = matches[
    matches.isnull().any(axis=1)
]

display(matches_with_missing.T)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38
match_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39
date,"March 28, 2026","March 29, 2026","March 30, 2026","March 31, 2026","April 01, 2026","April 02, 2026","April 03, 2026","April 04, 2026","April 04, 2026","April 05, 2026","April 05, 2026","April 06, 2026","April 07, 2026","April 08, 2026","April 09, 2026","April 10, 2026","April 11, 2026","April 11, 2026","April 12, 2026","April 12, 2026","April 13, 2026","April 14, 2026","April 15, 2026","April 16, 2026","April 17, 2026","April 18, 2026","April 18, 2026","April 19, 2026","April 19, 2026","April 20, 2026","April 21, 2026","April 22, 2026","April 23, 2026","April 24, 2026","April 25, 2026","April 25, 2026","April 26, 2026","April 26, 2026","April 27, 2026"
venue,"M. Chinnaswamy Stadium, Bangalore","Wankhede Stadium, Mumbai","Barsapara Stadium, Guwahati","New PCA Cricket Stadium, Mullanpur","Ekana Cricket Stadium, Lucknow","Eden Gardens, Kolkata","MA Chidambaram Stadium, Chennai","Arun Jaitley Stadium, Delhi","Narendra Modi Stadium, Ahmedabad","Rajiv Gandhi International Stadium, Hyderabad","M. Chinnaswamy Stadium, Bangalore","Eden Gardens, Kolkata","Barsapara Stadium, Guwahati","Arun Jaitley Stadium, Delhi","Eden Gardens, Kolkata","Barsapara Stadium, Guwahati","New PCA Cricket Stadium, Mullanpur","MA Chidambaram Stadium, Chennai","Ekana Cricket Stadium, Lucknow","Wankhede Stadium, Mumbai","Rajiv Gandhi International Stadium, Hyderabad","MA Chidambaram Stadium, Chennai","M. Chinnaswamy Stadium, Bangalore","Wankhede Stadium, Mumbai","Narendra Modi Stadium, Ahmedabad","M. Chinnaswamy Stadium, Bangalore","Rajiv Gandhi International Stadium, Hyderabad","Eden Gardens, Kolkata","New PCA Cricket Stadium, Mullanpur","Narendra Modi Stadium, Ahmedabad","Rajiv Gandhi International Stadium, Hyderabad","Ekana Cricket Stadium, Lucknow","Wankhede Stadium, Mumbai","M. Chinnaswamy Stadium, Bangalore","Arun Jaitley Stadium, Delhi","Sawai Mansingh Stadium, Jaipur","Narendra Modi Stadium, Ahmedabad","Ekana Cricket Stadium, Lucknow","Arun Jaitley Stadium, Delhi"
team1,RCB,MI,RR,PBKS,LSG,KKR,CSK,DC,GT,SRH,RCB,KKR,RR,DC,KKR,RR,PBKS,CSK,LSG,MI,SRH,CSK,RCB,MI,GT,RCB,SRH,KKR,PBKS,GT,SRH,LSG,MI,RCB,DC,RR,GT,LSG,DC
team2,SRH,KKR,CSK,GT,DC,SRH,PBKS,MI,RR,LSG,CSK,PBKS,MI,GT,LSG,RCB,SRH,DC,GT,RCB,RR,KKR,LSG,PBKS,KKR,DC,CSK,RR,LSG,MI,DC,RR,CSK,GT,PBKS,SRH,CSK,KKR,RCB
stage,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League,League
toss_winner,RCB,MI,RR,PBKS,DC,KKR,PBKS,DC,RR,LSG,CSK,KKR,MI,DC,LSG,RR,PBKS,DC,GT,MI,RR,KKR,RCB,PBKS,KKR,DC,CSK,RR,LSG,GT,DC,LSG,MI,RCB,DC,SRH,GT,LSG,RCB
toss_decision,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bat,Bowl,Bowl,Bat,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bat,Bowl,Bowl,Bat,Bowl,Bowl,Bowl,Bowl,Bowl,Bowl,Bat,Bowl,Bowl,Bowl,Bowl
first_ings_score,201.0,220.0,127.0,162.0,141.0,226.0,209.0,162.0,210.0,156.0,250.0,NaN,150.0,210.0,181.0,201.0,219.0,212.0,164.0,240.0,216.0,192.0,146.0,195.0,180.0,175.0,194.0,155.0,254.0,199.0,242.0,159.0,207.0,205.0,264.0,228.0,158.0,155.0,75.0
first_ings_wkts,9.0,4.0,10.0,6.0,10.0,8.0,5.0,6.0,6.0,9.0,3.0,NaN,3.0,4.0,4.0,8.0,6.0,2.0,8.0,4.0,6.0,5.0,10.0,6.0,10.0,8.0,9.0,9.0,7.0,5.0,2.0,6.0,6.0,3.0,2.0,6.0,7.0,7.0,10.0


### Missing Values in Match Dataset

The match-level dataset contains 39 records. The missing-value audit identified:

- Match 12 is marked as abandoned and has missing score, winner, and player-performance fields.
- `wb_runs` and `wb_wickets` contain many missing values, which may be expected because the fields represent different types of winning margins.
- The abandoned match's missing values should be retained rather than imputed.

**Conclusion:** The missing values in match 12 appear consistent with its abandoned status. Winning-margin fields require result-aware interpretation rather than blanket imputation.

## Step 6.3 — Check duplicate match IDs

In [39]:
print("Total match records:", len(matches))
print("Unique match IDs:", matches["match_id"].nunique())
print("Duplicate match IDs:", matches["match_id"].duplicated().sum())

Total match records: 39
Unique match IDs: 39
Duplicate match IDs: 0


## Check match-level data consistency

In [40]:
print("Date range:")
print("Start:", matches["date"].min())
print("End:", matches["date"].max())

print("\nUnique teams in matches:")
teams = sorted(set(matches["team1"].dropna()) | set(matches["team2"].dropna()))
print(teams)

print("\nMissing team1:", matches["team1"].isna().sum())
print("Missing team2:", matches["team2"].isna().sum())

Date range:
Start: April 01, 2026
End: March 31, 2026

Unique teams in matches:
['CSK', 'DC', 'GT', 'KKR', 'LSG', 'MI', 'PBKS', 'RCB', 'RR', 'SRH']

Missing team1: 0
Missing team2: 0


## Inspect match dates

In [41]:
matches[["match_id", "date", "team1", "team2"]].sort_values("date").to_string(index=False)

' match_id           date team1 team2\n        5 April 01, 2026   LSG    DC\n        6 April 02, 2026   KKR   SRH\n        7 April 03, 2026   CSK  PBKS\n        8 April 04, 2026    DC    MI\n        9 April 04, 2026    GT    RR\n       10 April 05, 2026   SRH   LSG\n       11 April 05, 2026   RCB   CSK\n       12 April 06, 2026   KKR  PBKS\n       13 April 07, 2026    RR    MI\n       14 April 08, 2026    DC    GT\n       15 April 09, 2026   KKR   LSG\n       16 April 10, 2026    RR   RCB\n       17 April 11, 2026  PBKS   SRH\n       18 April 11, 2026   CSK    DC\n       19 April 12, 2026   LSG    GT\n       20 April 12, 2026    MI   RCB\n       21 April 13, 2026   SRH    RR\n       22 April 14, 2026   CSK   KKR\n       23 April 15, 2026   RCB   LSG\n       24 April 16, 2026    MI  PBKS\n       25 April 17, 2026    GT   KKR\n       26 April 18, 2026   RCB    DC\n       27 April 18, 2026   SRH   CSK\n       29 April 19, 2026  PBKS   LSG\n       28 April 19, 2026   KKR    RR\n       30 A

## Check for duplicate matchups on the same date

In [42]:
matchups = matches.copy()

matchups["team_pair"] = matchups.apply(
    lambda row: " vs ".join(sorted([row["team1"], row["team2"]])),
    axis=1
)

duplicate_matchups = matchups[
    matchups.duplicated(subset=["date", "team_pair"], keep=False)
].sort_values(["date", "team_pair"])

print("Repeated team matchups on the same date:")
print(duplicate_matchups[["match_id", "date", "team_pair"]].to_string(index=False))

Repeated team matchups on the same date:
Empty DataFrame
Columns: [match_id, date, team_pair]
Index: []


## Check for teams playing themselves

In [43]:
same_team_matches = matches[matches["team1"] == matches["team2"]]

print("Matches where a team plays itself:", len(same_team_matches))
print(same_team_matches[["match_id", "date", "team1", "team2"]])

Matches where a team plays itself: 0
Empty DataFrame
Columns: [match_id, date, team1, team2]
Index: []


## Validate match results

In [44]:
valid_result = (
    matches["match_winner"].isna()
    | matches.apply(
        lambda row: row["match_winner"] in [row["team1"], row["team2"]]
        if pd.notna(row["match_winner"]) else True,
        axis=1
    )
)

invalid_results = matches[~valid_result]

print("Invalid match winners:", len(invalid_results))
print(invalid_results[["match_id", "team1", "team2", "match_winner"]])

Invalid match winners: 0
Empty DataFrame
Columns: [match_id, team1, team2, match_winner]
Index: []


## Validate the match scores

In [45]:
score_cols = ["first_ings_score", "second_ings_score"]
wicket_cols = ["first_ings_wkts", "second_ings_wkts"]

print("Negative scores:")
print(matches[matches[score_cols].lt(0).any(axis=1)][
    ["match_id"] + score_cols
])

print("\nNegative wickets:")
print(matches[matches[wicket_cols].lt(0).any(axis=1)][
    ["match_id"] + wicket_cols
])

print("\nWickets above 10:")
print(matches[matches[wicket_cols].gt(10).any(axis=1)][
    ["match_id"] + wicket_cols
])

Negative scores:
Empty DataFrame
Columns: [match_id, first_ings_score, second_ings_score]
Index: []

Negative wickets:
Empty DataFrame
Columns: [match_id, first_ings_wkts, second_ings_wkts]
Index: []

Wickets above 10:
Empty DataFrame
Columns: [match_id, first_ings_wkts, second_ings_wkts]
Index: []


### Score and Wicket Range Validation

- No negative innings scores were found.
- No negative wicket counts were found.
- No innings had more than 10 wickets.
- All checked score and wicket values passed the basic range validation.

**Conclusion:** No range violations were identified. Score consistency with the delivery-level dataset will be checked separately.

## Check innings scores against the deliveries data

In [48]:
# Calculate total runs from deliveries
delivery_scores = (
    deliveries.groupby(["match_no", "innings"])["calculated_runs"]
    .sum()
    .reset_index(name="runs_from_deliveries")
)

# Reshape innings into separate columns
score_check = delivery_scores.pivot(
    index="match_no",
    columns="innings",
    values="runs_from_deliveries"
).rename(columns={
    1: "delivery_first_score",
    2: "delivery_second_score"
}).reset_index()

# Compare with match-level scores
score_check = score_check.merge(
    matches[["match_id", "first_ings_score", "second_ings_score"]],
    left_on="match_no",
    right_on="match_id",
    how="left"
)

score_check["first_score_diff"] = (
    score_check["delivery_first_score"] - score_check["first_ings_score"]
)

score_check["second_score_diff"] = (
    score_check["delivery_second_score"] - score_check["second_ings_score"]
)

print(score_check.to_string(index=False))

 match_no  delivery_first_score  delivery_second_score   3   4  match_id  first_ings_score  second_ings_score  first_score_diff  second_score_diff
        1                 203.0                  203.0 NaN NaN       1.0             201.0              203.0               2.0                0.0
        2                 220.0                  224.0 NaN NaN       2.0             220.0              224.0               0.0                0.0
        3                 128.0                  128.0 NaN NaN       3.0             127.0              128.0               1.0                0.0
        4                 162.0                  165.0 NaN NaN       4.0             162.0              165.0               0.0                0.0
        5                 141.0                  144.0 NaN NaN       5.0             141.0              145.0               0.0               -1.0
        6                 226.0                  160.0 NaN NaN       6.0             226.0              161.0         

In [47]:
print(deliveries.columns.tolist())

['match_no', 'date', 'stage', 'venue', 'batting_team', 'bowling_team', 'innings', 'over', 'striker', 'bowler', 'runs_of_bat', 'extras', 'wide', 'legbyes', 'byes', 'noballs', 'wicket_type', 'player_dismissed', 'fielder', 'calculated_runs']


## Summarize the score differences

In [49]:
# Keep only matches that exist in both datasets
matched_scores = score_check[
    score_check["match_id"].notna()
].copy()

# Exclude missing match-level scores from the mismatch count
first_mismatch = matched_scores[
    matched_scores["first_score_diff"].notna()
    & (matched_scores["first_score_diff"] != 0)
]

second_mismatch = matched_scores[
    matched_scores["second_score_diff"].notna()
    & (matched_scores["second_score_diff"] != 0)
]

print("First-innings mismatches:", len(first_mismatch))
print(first_mismatch[["match_no", "first_score_diff"]].to_string(index=False))

print("\nSecond-innings mismatches:", len(second_mismatch))
print(second_mismatch[["match_no", "second_score_diff"]].to_string(index=False))

print("\nDelivery match numbers without match-table records:")
print(sorted(set(deliveries["match_no"].unique()) - set(matches["match_id"].unique())))

First-innings mismatches: 11
 match_no  first_score_diff
        1               2.0
        3               1.0
       17              -3.0
       20              -1.0
       21              -2.0
       22              -1.0
       28              -2.0
       29              -1.0
       36              -1.0
       37              -1.0
       38              -1.0

Second-innings mismatches: 9
 match_no  second_score_diff
        5               -1.0
        6               -1.0
        7               -1.0
       14               -1.0
       27               -1.0
       28                1.0
       31               -2.0
       33               -1.0
       35               -1.0

Delivery match numbers without match-table records:
[np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(

## Validate calculated_runs

In [50]:
# Compare calculated runs with batting runs plus extras
deliveries["expected_runs"] = (
    deliveries["runs_of_bat"] + deliveries["extras"]
)

deliveries["run_difference"] = (
    deliveries["calculated_runs"] - deliveries["expected_runs"]
)

run_mismatches = deliveries[deliveries["run_difference"] != 0]

print("Total deliveries:", len(deliveries))
print("Run calculation mismatches:", len(run_mismatches))

print(
    run_mismatches[
        ["match_no", "innings", "over", "runs_of_bat",
         "extras", "calculated_runs", "run_difference"]
    ].head(20).to_string(index=False)
)

Total deliveries: 17477
Run calculation mismatches: 0
Empty DataFrame
Columns: [match_no, innings, over, runs_of_bat, extras, calculated_runs, run_difference]
Index: []


## Investigate the mismatched innings

In [51]:
match17 = deliveries[
    (deliveries["match_no"] == 17) &
    (deliveries["innings"] == 1)
]

print("Runs from deliveries:", match17["calculated_runs"].sum())
print("Batting runs:", match17["runs_of_bat"].sum())
print("Extras:", match17["extras"].sum())

print("\nExtras breakdown:")
print(
    match17[["wide", "legbyes", "byes", "noballs"]]
    .sum()
)

print("\nMatch-table score:")
print(
    matches.loc[
        matches["match_id"] == 17,
        ["first_ings_score", "first_ings_wkts"]
    ]
)

Runs from deliveries: 216
Batting runs: 206
Extras: 10

Extras breakdown:
wide       8
legbyes    0
byes       1
noballs    1
dtype: int64

Match-table score:
    first_ings_score  first_ings_wkts
16             219.0              6.0


## Check whether any deliveries are missing from the innings

In [52]:
print("Number of delivery records:", len(match17))
print("Unique over labels:", sorted(match17["over"].unique()))

print("\nDelivery count per over:")
print(match17.groupby("over").size().to_string())

print("\nLast 10 delivery records:")
print(
    match17[
        ["over", "striker", "bowler", "runs_of_bat", "extras",
         "calculated_runs", "wicket_type"]
    ].tail(10).to_string(index=False)
)

Number of delivery records: 129
Unique over labels: [np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(1.6), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(2.5), np.float64(2.6), np.float64(3.1), np.float64(3.2), np.float64(3.3), np.float64(3.4), np.float64(3.5), np.float64(3.6), np.float64(4.1), np.float64(4.2), np.float64(4.3), np.float64(4.4), np.float64(4.5), np.float64(4.6), np.float64(5.1), np.float64(5.2), np.float64(5.3), np.float64(5.4), np.float64(5.5), np.float64(5.6), np.float64(6.1), np.float64(6.2), np.float64(6.3), np.float64(6.4), np.float64(6.5), np.float64(6.6), np.float64(7.1), np.float64(7.2), np.float64(7.3), np.float64(7.4), np.float64(7.5), np.float64(7.6), np.float64(8.1), np.float64(8.2), np.float64(8.3), np.float64(8.4), np.float64(8.5), np.float64(8.6), np.float64(9.1), np.float64(9

## Inspect the extra deliveries

In [53]:
# Find over labels with more than one delivery record
extra_records = (
    match17.groupby("over")
    .filter(lambda group: len(group) > 1)
    .sort_values("over")
)

print(
    extra_records[
        ["over", "striker", "bowler", "runs_of_bat",
         "extras", "wide", "noballs", "calculated_runs"]
    ].to_string(index=False)
)

 over          striker           bowler  runs_of_bat  extras  wide  noballs  calculated_runs
  0.1  Abhishek Sharma   Arshdeep Singh            0       1     1        0                1
  0.1  Abhishek Sharma   Arshdeep Singh            4       0     0        0                4
  0.3      Travis Head   Arshdeep Singh            0       0     0        0                0
  0.3      Travis Head   Arshdeep Singh            0       1     1        0                1
  2.3  Abhishek Sharma   Arshdeep Singh            0       0     0        0                0
  2.3  Abhishek Sharma   Arshdeep Singh            0       1     1        0                1
  2.4  Abhishek Sharma   Arshdeep Singh            0       1     1        0                1
  2.4  Abhishek Sharma   Arshdeep Singh            6       0     0        0                6
  5.4      Travis Head  Xavier Bartlett            1       0     0        0                1
  5.4      Travis Head  Xavier Bartlett            0       1     0    

## Check whether Match 17 has a Super Over or additional innings

In [54]:
print(
    deliveries[deliveries["match_no"] == 17]
    .groupby("innings")
    .agg(
        delivery_records=("calculated_runs", "size"),
        total_runs=("calculated_runs", "sum")
    )
)

print("\nMatch 17 metadata:")
print(
    matches.loc[
        matches["match_id"] == 17
    ].to_string(index=False)
)

         delivery_records  total_runs
innings                              
1                     129         216
2                     118         223

Match 17 metadata:
 match_id           date                              venue team1 team2  stage toss_winner toss_decision  first_ings_score  first_ings_wkts  second_ings_score  second_ings_wkts match_result match_winner  wb_runs  wb_wickets  balls_left player_of_the_match      top_scorer  highscore  best_bowling best_bowling_figure super_over_match
       17 April 11, 2026 New PCA Cricket Stadium, Mullanpur  PBKS   SRH League        PBKS          Bowl             219.0              6.0              223.0               4.0    completed         PBKS      NaN         6.0         7.0        Shreyas Iyer Abhishek Sharma       74.0 Shivang Kumar               3--33               No


In [55]:
match17_innings1 = deliveries[
    (deliveries["match_no"] == 17) &
    (deliveries["innings"] == 1)
]

print("Run components:")
print(
    match17_innings1[
        ["runs_of_bat", "extras", "wide", "legbyes", "byes", "noballs"]
    ].sum()
)

print("\nWicket-type counts:")
print(match17_innings1["wicket_type"].value_counts(dropna=False))

Run components:
runs_of_bat    206
extras          10
wide             8
legbyes          0
byes             1
noballs          1
dtype: int64

Wicket-type counts:
wicket_type
NaN       123
caught      5
runout      1
Name: count, dtype: int64


## Document this finding

### Match 17: Score Consistency Investigation

- First-innings delivery total: 216
- Match-table first-innings score: 219
- Difference: 3 runs
- Runs from bat: 206
- Extras: 10 (8 wides, 1 bye, 1 no-ball)
- Recorded wickets: 6
- Match-table wickets: 6
- Second-innings delivery and match-table scores: both 223
- Match-table winner: PBKS, although the listed scores show SRH scoring more runs.
- Super Over: No, according to the match metadata.

**Conclusion:** The delivery-level run components and wicket count are internally consistent, but the first-innings score and reported winner require further verification. No source values were changed.

## Build a diagnostic table

In [56]:
# Summarize delivery-level innings data
innings_audit = (
    deliveries.groupby(["match_no", "innings"])
    .agg(
        delivery_runs=("calculated_runs", "sum"),
        bat_runs=("runs_of_bat", "sum"),
        extras=("extras", "sum"),
        wides=("wide", "sum"),
        legbyes=("legbyes", "sum"),
        byes=("byes", "sum"),
        noballs=("noballs", "sum"),
        wickets=("player_dismissed", "count"),
        records=("calculated_runs", "size")
    )
    .reset_index()
)

# Split the innings summaries into first and second innings
first = innings_audit[innings_audit["innings"] == 1].drop(
    columns="innings"
).add_prefix("first_")

second = innings_audit[innings_audit["innings"] == 2].drop(
    columns="innings"
).add_prefix("second_")

# Merge with match-level data
diagnostic = matches[
    ["match_id", "first_ings_score", "second_ings_score",
     "first_ings_wkts", "second_ings_wkts", "match_winner",
     "team1", "team2"]
].merge(
    first, left_on="match_id", right_on="first_match_no", how="left"
).merge(
    second, left_on="match_id", right_on="second_match_no", how="left"
)

# Calculate differences
diagnostic["first_run_diff"] = (
    diagnostic["first_delivery_runs"] - diagnostic["first_ings_score"]
)
diagnostic["second_run_diff"] = (
    diagnostic["second_delivery_runs"] - diagnostic["second_ings_score"]
)

# Show matches with score discrepancies
mismatches = diagnostic[
    (diagnostic["first_run_diff"].fillna(0) != 0) |
    (diagnostic["second_run_diff"].fillna(0) != 0)
]

print("Mismatched matches:", len(mismatches))
print(
    mismatches[
        ["match_id", "team1", "team2",
         "first_delivery_runs", "first_ings_score", "first_run_diff",
         "second_delivery_runs", "second_ings_score", "second_run_diff",
         "first_wickets", "second_wickets", "match_winner"]
    ].to_string(index=False)
)

Mismatched matches: 19
 match_id team1 team2  first_delivery_runs  first_ings_score  first_run_diff  second_delivery_runs  second_ings_score  second_run_diff  first_wickets  second_wickets match_winner
        1   RCB   SRH                  203             201.0             2.0                 203.0              203.0              0.0              9             4.0          RCB
        3    RR   CSK                  128             127.0             1.0                 128.0              128.0              0.0             10             2.0           RR
        5   LSG    DC                  141             141.0             0.0                 144.0              145.0             -1.0             10             4.0           DC
        6   KKR   SRH                  226             226.0             0.0                 160.0              161.0             -1.0              8            10.0          SRH
        7   CSK  PBKS                  209             209.0             0.0      

## Check wicket-count discrepancies

In [57]:
# Compare wicket counts from deliveries with match-level wicket counts
diagnostic["first_wicket_diff"] = (
    diagnostic["first_wickets"] - diagnostic["first_ings_wkts"]
)

diagnostic["second_wicket_diff"] = (
    diagnostic["second_wickets"] - diagnostic["second_ings_wkts"]
)

wicket_mismatches = diagnostic[
    (diagnostic["first_wicket_diff"].fillna(0) != 0) |
    (diagnostic["second_wicket_diff"].fillna(0) != 0)
]

print("Matches with wicket-count mismatches:", len(wicket_mismatches))

print(
    wicket_mismatches[
        ["match_id", "team1", "team2",
         "first_wickets", "first_ings_wkts", "first_wicket_diff",
         "second_wickets", "second_ings_wkts", "second_wicket_diff"]
    ].to_string(index=False)
)

Matches with wicket-count mismatches: 6
 match_id team1 team2  first_wickets  first_ings_wkts  first_wicket_diff  second_wickets  second_ings_wkts  second_wicket_diff
       18   CSK    DC              1              2.0               -1.0            10.0              10.0                 0.0
       20    MI   RCB              4              4.0                0.0             6.0               5.0                 1.0
       21   SRH    RR              5              6.0               -1.0            10.0              10.0                 0.0
       26   RCB    DC              7              8.0               -1.0             4.0               4.0                 0.0
       28   KKR    RR              8              9.0               -1.0             5.0               6.0                -1.0
       37    GT   CSK              7              7.0                0.0             1.0               2.0                -1.0


## Check the match results against the delivery totals

In [58]:
result_check = matches[
    matches["match_winner"].notna()
].copy()

result_check["score_winner"] = result_check.apply(
    lambda row: (
        row["team1"]
        if row["first_ings_score"] > row["second_ings_score"]
        else row["team2"]
        if row["second_ings_score"] > row["first_ings_score"]
        else "Tie"
    ),
    axis=1
)

result_mismatches = result_check[
    result_check["score_winner"] != result_check["match_winner"]
]

print("Matches with result-score mismatches:", len(result_mismatches))

print(
    result_mismatches[
        ["match_id", "team1", "team2",
         "first_ings_score", "second_ings_score",
         "match_winner", "score_winner"]
    ].to_string(index=False)
)

Matches with result-score mismatches: 19
 match_id team1 team2  first_ings_score  second_ings_score match_winner score_winner
        1   RCB   SRH             201.0              203.0          RCB          SRH
        2    MI   KKR             220.0              224.0           MI          KKR
        3    RR   CSK             127.0              128.0           RR          CSK
        4  PBKS    GT             162.0              165.0         PBKS           GT
        6   KKR   SRH             226.0              161.0          SRH          KKR
        8    DC    MI             162.0              164.0           DC           MI
        9    GT    RR             210.0              204.0           RR           GT
       14    DC    GT             210.0              209.0           GT           DC
       16    RR   RCB             201.0              202.0           RR          RCB
       17  PBKS   SRH             219.0              223.0         PBKS          SRH
       20    MI   RCB   

## Classify the result discrepancies

In [59]:
result_review = result_mismatches.merge(
    diagnostic[
        ["match_id",
         "first_delivery_runs", "second_delivery_runs",
         "first_run_diff", "second_run_diff"]
    ],
    on="match_id",
    how="left"
)

print(
    result_review[
        ["match_id", "team1", "team2",
         "first_ings_score", "first_delivery_runs",
         "second_ings_score", "second_delivery_runs",
         "match_winner", "score_winner"]
    ].to_string(index=False)
)

 match_id team1 team2  first_ings_score  first_delivery_runs  second_ings_score  second_delivery_runs match_winner score_winner
        1   RCB   SRH             201.0                  203              203.0                 203.0          RCB          SRH
        2    MI   KKR             220.0                  220              224.0                 224.0           MI          KKR
        3    RR   CSK             127.0                  128              128.0                 128.0           RR          CSK
        4  PBKS    GT             162.0                  162              165.0                 165.0         PBKS           GT
        6   KKR   SRH             226.0                  226              161.0                 160.0          SRH          KKR
        8    DC    MI             162.0                  162              164.0                 164.0           DC           MI
        9    GT    RR             210.0                  210              204.0                 204.0   

## Create a consolidated match-quality summary

In [61]:
score_mismatches = diagnostic[
    (diagnostic["first_run_diff"].fillna(0) != 0) |
    (diagnostic["second_run_diff"].fillna(0) != 0)
]

print("MATCH DATA QUALITY SUMMARY")
print("-" * 40)

print("Total match records:", len(matches))
print("Unique match IDs:", matches["match_id"].nunique())
print("Matches with score mismatches:", len(score_mismatches))
print("Matches with wicket-count mismatches:", len(wicket_mismatches))
print("Matches with winner-score mismatches:", len(result_mismatches))
print("Matches missing a winner:", matches["match_winner"].isna().sum())

print("\nMatches requiring result review:")
print(sorted(result_mismatches["match_id"].astype(int).tolist()))

MATCH DATA QUALITY SUMMARY
----------------------------------------
Total match records: 39
Unique match IDs: 39
Matches with score mismatches: 19
Matches with wicket-count mismatches: 6
Matches with winner-score mismatches: 19
Matches missing a winner: 1

Matches requiring result review:
[1, 2, 3, 4, 6, 8, 9, 14, 16, 17, 20, 23, 25, 28, 30, 33, 34, 37, 38]


In [62]:
match_quality = matches[
    ["match_id", "team1", "team2", "match_winner"]
].copy()

score_issue_ids = set(score_mismatches["match_id"])
wicket_issue_ids = set(wicket_mismatches["match_id"])
result_issue_ids = set(result_mismatches["match_id"])

match_quality["score_issue"] = (
    match_quality["match_id"].isin(score_issue_ids)
)

match_quality["wicket_issue"] = (
    match_quality["match_id"].isin(wicket_issue_ids)
)

match_quality["result_issue"] = (
    match_quality["match_id"].isin(result_issue_ids)
)

match_quality["missing_winner"] = (
    match_quality["match_winner"].isna()
)

match_quality["needs_review"] = match_quality[
    ["score_issue", "wicket_issue", "result_issue", "missing_winner"]
].any(axis=1)

print("Matches flagged for review:", match_quality["needs_review"].sum())

display(
    match_quality[match_quality["needs_review"]]
    .sort_values("match_id")
    .reset_index(drop=True)
)

Matches flagged for review: 31


,match_id,team1,team2,match_winner,score_issue,wicket_issue,result_issue,missing_winner,needs_review
0,1,RCB,SRH,RCB,True,False,True,False,True
1,2,MI,KKR,MI,False,False,True,False,True
2,3,RR,CSK,RR,True,False,True,False,True
3,4,PBKS,GT,PBKS,False,False,True,False,True
4,5,LSG,DC,DC,True,False,False,False,True
5,6,KKR,SRH,SRH,True,False,True,False,True
6,7,CSK,PBKS,PBKS,True,False,False,False,True
7,8,DC,MI,DC,False,False,True,False,True
8,9,GT,RR,RR,False,False,True,False,True
9,12,KKR,PBKS,NaN,False,False,False,True,True


## Save the audit findings

In [63]:
issue_columns = [
    "score_issue",
    "wicket_issue",
    "result_issue",
    "missing_winner"
]

print("ISSUE COUNTS")
print("-" * 30)

for col in issue_columns:
    print(f"{col}: {match_quality[col].sum()}")

print("\nCOMBINATION OF ISSUES")
print("-" * 30)

print(
    match_quality.groupby(issue_columns, dropna=False)
    .size()
    .reset_index(name="match_count")
    .sort_values("match_count", ascending=False)
    .to_string(index=False)
)

ISSUE COUNTS
------------------------------
score_issue: 19
wicket_issue: 6
result_issue: 19
missing_winner: 1

COMBINATION OF ISSUES
------------------------------
 score_issue  wicket_issue  result_issue  missing_winner  match_count
       False         False          True           False            9
       False         False         False           False            8
        True         False         False           False            8
        True         False          True           False            7
        True          True          True           False            3
       False          True         False           False            2
       False         False         False            True            1
        True          True         False           False            1


## Step 1: Inspect the batting statistics dataset

In [5]:
import pandas as pd
from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select batting statistics CSV",
    filetypes=[("CSV files", "*.csv")]
)

if file_path:
    batting_stats = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    print("Shape:", batting_stats.shape)
    display(batting_stats.head())
else:
    print("No file selected.")

Dataset loaded successfully!
Shape: (15, 17)


,position,batsman,team,matches,innings,runs,impact,average,strike_rate,not_outs,high_score,balls_faced,hundreds,fifties,ducks,fours,sixes
0,1,Abhishek Sharma,SRH,8,8,380,395.6,54.28,212.29,1,135,179,1,3,2,36,28
1,2,KL Rahul,DC,8,8,358,317.6,51.14,185.49,1,152,193,1,2,1,36,19
2,3,Vaibhav Sooryavanshi,RR,8,8,357,433.2,44.62,234.86,0,103,152,1,2,1,31,32
3,4,Virat Kohli,RCB,8,8,351,282.2,58.50,162.50,2,81,216,0,3,0,37,14
4,5,Heinrich Klaasen,SRH,8,8,349,258.6,49.85,149.78,1,62,233,0,3,0,25,14


## Step 2 :Check data validity

In [6]:
# Basic numerical ranges
print("Matches:", batting_stats["matches"].min(),
      "to", batting_stats["matches"].max())

print("Innings:", batting_stats["innings"].min(),
      "to", batting_stats["innings"].max())

print("Runs:", batting_stats["runs"].min(),
      "to", batting_stats["runs"].max())

print("Strike rate:", batting_stats["strike_rate"].min(),
      "to", batting_stats["strike_rate"].max())

# Logical validation
print("\nLOGICAL VALIDATION")
print("-" * 35)

print("Innings > matches:",
      (batting_stats["innings"] > batting_stats["matches"]).sum())

print("Not outs > innings:",
      (batting_stats["not_outs"] > batting_stats["innings"]).sum())

print("High score > total runs:",
      (batting_stats["high_score"] > batting_stats["runs"]).sum())

print("Invalid negative values:",
      (batting_stats.select_dtypes(include="number") < 0).sum().sum())

Matches: 6 to 8
Innings: 6 to 8
Runs: 242 to 380
Strike rate: 148.64 to 249.01

LOGICAL VALIDATION
-----------------------------------
Innings > matches: 0
Not outs > innings: 0
High score > total runs: 0
Invalid negative values: 0


## Step 3: Verify batting average and strike rate

In [7]:
# Recalculate batting average
batting_stats["calculated_average"] = (
    batting_stats["runs"] /
    (batting_stats["innings"] - batting_stats["not_outs"])
)

# Recalculate strike rate
batting_stats["calculated_strike_rate"] = (
    batting_stats["runs"] /
    batting_stats["balls_faced"] * 100
)

# Compare with reported values
batting_stats["average_diff"] = (
    batting_stats["average"] -
    batting_stats["calculated_average"]
).round(2)

batting_stats["strike_rate_diff"] = (
    batting_stats["strike_rate"] -
    batting_stats["calculated_strike_rate"]
).round(2)

display(
    batting_stats[
        ["batsman", "average", "calculated_average",
         "average_diff", "strike_rate", "calculated_strike_rate",
         "strike_rate_diff"]
    ]
)

,batsman,average,calculated_average,average_diff,strike_rate,calculated_strike_rate,strike_rate_diff
0,Abhishek Sharma,54.28,54.285714,-0.01,212.29,212.290503,-0.00
1,KL Rahul,51.14,51.142857,-0.00,185.49,185.492228,-0.00
2,Vaibhav Sooryavanshi,44.62,44.625000,-0.01,234.86,234.868421,-0.01
3,Virat Kohli,58.50,58.500000,0.00,162.50,162.500000,0.00
4,Heinrich Klaasen,49.85,49.857143,-0.01,149.78,149.785408,-0.01
5,Shubman Gill,47.14,47.142857,-0.00,148.64,148.648649,-0.01
6,Sai Sudarshan,40.25,40.250000,0.00,163.45,163.451777,-0.00
7,Ishan Kishan,39.00,39.000000,0.00,198.72,198.726115,-0.01
8,Sanju Samson,50.66,50.666667,-0.01,169.83,169.832402,-0.00
9,Prabhsimran Singh,57.40,57.400000,0.00,192.61,192.617450,-0.01


## Step 4: Check player uniqueness and batting records

In [8]:
print("PLAYER UNIQUENESS")
print("-" * 35)

print("Unique player names:", batting_stats["batsman"].nunique())
print("Duplicate player names:",
      batting_stats["batsman"].duplicated().sum())

print("\nBOUNDARY AND MILESTONE CHECKS")
print("-" * 35)

print("Boundary runs exceed total runs:",
      ((batting_stats["fours"] * 4 +
        batting_stats["sixes"] * 6) >
       batting_stats["runs"]).sum())

print("Centuries greater than innings:",
      (batting_stats["hundreds"] > batting_stats["innings"]).sum())

print("Fifties greater than innings:",
      (batting_stats["fifties"] > batting_stats["innings"]).sum())

print("Ducks greater than innings:",
      (batting_stats["ducks"] > batting_stats["innings"]).sum())

print("\nPlayers with repeated names:")
display(
    batting_stats[
        batting_stats["batsman"].duplicated(keep=False)
    ][["batsman", "team", "matches", "runs"]]
)

PLAYER UNIQUENESS
-----------------------------------
Unique player names: 15
Duplicate player names: 0

BOUNDARY AND MILESTONE CHECKS
-----------------------------------
Boundary runs exceed total runs: 0
Centuries greater than innings: 0
Fifties greater than innings: 0
Ducks greater than innings: 0

Players with repeated names:


,batsman,team,matches,runs


## check the impact column

In [9]:
# Inspect the impact metric
print("IMPACT COLUMN SUMMARY")
print("-" * 35)

print(batting_stats["impact"].describe())

print("\nTop 5 players by impact:")
display(
    batting_stats.nlargest(5, "impact")[
        ["batsman", "team", "runs", "average",
         "strike_rate", "impact"]
    ]
)

print("\nCorrelation with other batting metrics:")
display(
    batting_stats[
        ["impact", "runs", "average", "strike_rate"]
    ].corr().round(2)
)

IMPACT COLUMN SUMMARY
-----------------------------------
count     15.000000
mean     308.680000
std       65.499871
min      233.400000
25%      251.800000
50%      282.200000
75%      366.550000
max      433.200000
Name: impact, dtype: float64

Top 5 players by impact:


,batsman,team,runs,average,strike_rate,impact
2,Vaibhav Sooryavanshi,RR,357,44.62,234.86,433.2
0,Abhishek Sharma,SRH,380,54.28,212.29,395.6
7,Ishan Kishan,SRH,312,39.00,198.72,381.7
8,Sanju Samson,CSK,304,50.66,169.83,369.4
13,Priyansh Arya,PBKS,254,42.33,249.01,363.7



Correlation with other batting metrics:


,impact,runs,average,strike_rate
impact,1.00,0.45,-0.06,0.73
runs,0.45,1.00,0.26,-0.01
average,-0.06,0.26,1.00,-0.02
strike_rate,0.73,-0.01,-0.02,1.00



### Audit Conclusion

The uploaded batting statistics dataset passed the internal validation checks performed during this audit. No missing values, exact duplicate rows, or logical inconsistencies were identified. The calculated batting averages and strike rates were also consistent with the provided figures, allowing for rounding differences.

No data corrections are currently required based on these checks. However, the dataset contains only 15 player records, and the statistics have not been independently verified against match-by-match batting data or an authoritative source.

**Decision:** Retain the original data without modification and document these limitations before proceeding to further analysis.

## Bowling statistics audit — Step 1: Load the bowling dataset

In [11]:
import pandas as pd
from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select bowling_stats.csv",
    filetypes=[("CSV files", "*.csv")]
)

if file_path:
    bowling_stats = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    print("Shape:", bowling_stats.shape)
    display(bowling_stats.head())
else:
    print("No file selected.")

Dataset loaded successfully!
Shape: (15, 17)


,position,bowler,team,matches,innings,wickets,economy,impact,bbi,overs,balls,dot_balls,maiden_overs,runs,avg,4wh,5wh
0,1,Bhuvneshwar Kumar,RCB,8,8,14,7.61,351.0,3--5,31.0,186,72,0,236,16.85,0,0
1,2,Anshul Kamboj,CSK,8,8,14,8.56,284.8,3--22,27.4,166,75,0,237,16.92,0,0
2,3,Eshan Malinga,SRH,8,8,14,9.44,293.1,4--32,27.0,162,61,0,255,18.21,1,0
3,4,Jofra Archer,RR,8,8,13,8.03,343.0,3--20,29.0,174,88,1,233,17.92,0,0
4,5,Prince Yadav,LSG,8,8,13,8.06,355.8,3--32,30.0,180,77,0,242,18.61,0,0


In [13]:
print(bowling_stats.columns.tolist())

['position', 'bowler', 'team', 'matches', 'innings', 'wickets', 'economy', 'impact', 'bbi', 'overs', 'balls', 'dot_balls', 'maiden_overs', 'runs', 'avg', '4wh', '5wh', 'calculated_avg', 'calculated_economy', 'calculated_balls']


## Basic Logical Consistency Checks

In [14]:
# Basic logical consistency checks

print("Dataset shape:", bowling_stats.shape)

# 1. Innings should not exceed matches
print("Innings greater than matches:",
      (bowling_stats["innings"] > bowling_stats["matches"]).sum())

# 2. Check for negative numeric values
numeric_cols = bowling_stats.select_dtypes(include="number").columns

print("\nNegative values by column:")
print((bowling_stats[numeric_cols] < 0).sum())

# 3. Four- and five-wicket hauls should not exceed innings
print("\n4-wicket hauls greater than innings:",
      (bowling_stats["4wh"] > bowling_stats["innings"]).sum())

print("5-wicket hauls greater than innings:",
      (bowling_stats["5wh"] > bowling_stats["innings"]).sum())

# 4. Check whether wickets exceed innings * 10
print("\nWickets greater than 10 times innings:",
      (bowling_stats["wickets"] > bowling_stats["innings"] * 10).sum())

# 5. Check for zero wickets with nonzero runs
print("\nZero wickets with nonzero runs:")
display(bowling_stats[
    (bowling_stats["wickets"] == 0) &
    (bowling_stats["runs"] > 0)
][["bowler", "wickets", "runs", "avg"]])

Dataset shape: (15, 20)
Innings greater than matches: 0

Negative values by column:
position              0
matches               0
innings               0
wickets               0
economy               0
impact                0
overs                 0
balls                 0
dot_balls             0
maiden_overs          0
runs                  0
avg                   0
4wh                   0
5wh                   0
calculated_avg        0
calculated_economy    0
calculated_balls      0
dtype: int64

4-wicket hauls greater than innings: 0
5-wicket hauls greater than innings: 0

Wickets greater than 10 times innings: 0

Zero wickets with nonzero runs:


,bowler,wickets,runs,avg


##  Step 2: Validate Bowling Calculations

In this step, we recalculate the bowling average, economy rate, and total balls bowled using the available data. We then compare these calculated values with the original dataset to identify any inconsistencies.

In [16]:
# Recalculate bowling average
bowling_stats["calculated_avg"] = (
    bowling_stats["runs"] /
    bowling_stats["wickets"].replace(0, float("nan"))
).round(2)

# Recalculate economy rate
bowling_stats["calculated_economy"] = (
    bowling_stats["runs"] / (bowling_stats["balls"] / 6)
).round(2)

# Convert overs notation into total balls
bowling_stats["calculated_balls"] = (
    bowling_stats["overs"].astype(int) * 6
    + (bowling_stats["overs"] % 1 * 10).round().astype(int)
)

# Compare calculated and original values
print("Bowling average mismatches:",
      ((bowling_stats["calculated_avg"] -
        bowling_stats["avg"]).abs() > 0.02).sum())

print("Economy mismatches:",
      ((bowling_stats["calculated_economy"] -
        bowling_stats["economy"]).abs() > 0.02).sum())

print("Overs and balls mismatches:",
      (bowling_stats["calculated_balls"] !=
       bowling_stats["balls"]).sum())

Bowling average mismatches: 0
Economy mismatches: 0
Overs and balls mismatches: 0


### Step 3: Additional Bowling Metric Checks

In this step, we examine additional bowling statistics to identify unusual or logically inconsistent values. We validate dot balls and maiden overs against the total balls and overs bowled, and inspect records of four- and five-wicket hauls.

In [17]:
# Step 3: Additional bowling metric checks

# 1. Check dot balls against total balls
print("Dot balls greater than total balls:",
      (bowling_stats["dot_balls"] > bowling_stats["balls"]).sum())

# 2. Check maiden overs against total overs
print("Maiden overs greater than overs bowled:",
      (bowling_stats["maiden_overs"] > bowling_stats["overs"]).sum())

# 3. Inspect four-wicket and five-wicket hauls
haul_records = bowling_stats[
    (bowling_stats["4wh"] > 0) |
    (bowling_stats["5wh"] > 0)
]

print("\nFour- and five-wicket haul records:")
display(
    haul_records[
        ["bowler", "team", "innings", "wickets", "4wh", "5wh"]
    ]
)

Dot balls greater than total balls: 0
Maiden overs greater than overs bowled: 0

Four- and five-wicket haul records:


,bowler,team,innings,wickets,4wh,5wh
2,Eshan Malinga,SRH,8,14,1,0
6,Prasidh Krishna,GT,7,12,1,0
7,Ravi Bishnoi,RR,8,11,1,0
8,Mohsin Khan,LSG,4,9,0,1
11,Jamie Overton,CSK,7,9,1,0
13,Josh Hazlewood,RCB,5,8,1,0


### Step 4: Statistical Summary

In this step, we use descriptive statistics to understand the distribution of key bowling performance metrics, including wickets, economy rate, bowling average, overs, and dot balls.

In [18]:
# Step 4: Statistical summary

summary_cols = [
    "matches",
    "innings",
    "wickets",
    "economy",
    "impact",
    "overs",
    "balls",
    "dot_balls",
    "maiden_overs",
    "runs",
    "avg",
    "4wh",
    "5wh"
]

display(bowling_stats[summary_cols].describe().round(2))

,matches,innings,wickets,economy,impact,overs,balls,dot_balls,maiden_overs,runs,avg,4wh,5wh
count,15.00,15.00,15.00,15.00,15.00,15.00,15.00,15.00,15.00,15.00,15.00,15.00,15.00
mean,7.40,7.27,11.00,8.78,275.33,26.05,156.47,62.60,0.27,230.20,21.44,0.33,0.07
std,1.24,1.22,2.36,1.12,59.68,4.56,27.26,16.33,0.80,50.01,5.54,0.49,0.26
min,4.00,4.00,8.00,6.37,188.20,16.00,96.00,37.00,0.00,102.00,11.33,0.00,0.00
25%,7.50,7.00,9.00,7.98,230.40,25.00,150.00,51.00,0.00,229.50,18.07,0.00,0.00
50%,8.00,8.00,11.00,8.85,273.30,27.00,162.00,61.00,0.00,237.00,20.55,0.00,0.00
75%,8.00,8.00,13.00,9.58,318.05,29.00,174.00,76.00,0.00,257.50,24.58,1.00,0.00
max,8.00,8.00,14.00,10.34,385.10,31.00,186.00,88.00,3.00,294.00,32.00,1.00,1.00


In [19]:
# Display summary with metrics as rows
display(
    bowling_stats[summary_cols]
    .describe()
    .round(2)
    .T
)

,count,mean,std,min,25%,50%,75%,max
matches,15.0,7.40,1.24,4.00,7.50,8.00,8.00,8.00
innings,15.0,7.27,1.22,4.00,7.00,8.00,8.00,8.00
wickets,15.0,11.00,2.36,8.00,9.00,11.00,13.00,14.00
economy,15.0,8.78,1.12,6.37,7.98,8.85,9.58,10.34
impact,15.0,275.33,59.68,188.20,230.40,273.30,318.05,385.10
overs,15.0,26.05,4.56,16.00,25.00,27.00,29.00,31.00
balls,15.0,156.47,27.26,96.00,150.00,162.00,174.00,186.00
dot_balls,15.0,62.60,16.33,37.00,51.00,61.00,76.00,88.00
maiden_overs,15.0,0.27,0.80,0.00,0.00,0.00,0.00,3.00
runs,15.0,230.20,50.01,102.00,229.50,237.00,257.50,294.00


### Statistical Summary — Key Observations

- The dataset contains 15 bowling records, with an average of 11 wickets per bowler.
- The average economy rate is 8.78, with values ranging from 6.37 to 10.34.
- Bowlers conceded an average of 230.2 runs and bowled approximately 26 overs each.
- The average bowling average is 21.44, with a minimum of 11.33 and a maximum of 32.00.
- Impact scores range from 188.20 to 385.10, indicating variation among the selected bowlers.
- Most bowlers have no recorded four- or five-wicket haul in this dataset.

**Limitation:** These observations describe only the 15 available records. They should not be generalized to the complete tournament without confirming dataset coverage.

### Bowling Statistics Audit — Final Conclusion

The bowling statistics dataset contains 15 records and passed the internal validation checks performed during the audit.

- No missing values or exact duplicate rows were found in the initial inspection.
- No logical inconsistencies were identified in the checks for innings, negative values, wickets, and wicket-haul counts.
- Recalculated bowling averages, economy rates, and overs-to-balls conversions matched the supplied figures within the selected tolerance.
- Dot-ball and maiden-over checks revealed no inconsistencies.
- Descriptive statistics were generated to understand the distribution of bowling performance metrics.

**Decision:** No data corrections are currently justified by the checks performed. The original data is retained without modification.

**Limitation:** The dataset contains only 15 bowling records. The statistics have not been independently verified against complete ball-by-ball data or an authoritative source.

## Step 1 :Load the fielding dataset

In [21]:
import pandas as pd
from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select fielding_stats.csv",
    filetypes=[("CSV files", "*.csv")]
)

if file_path:
    fielding_stats = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    print("Shape:", fielding_stats.shape)
    display(fielding_stats.head())
else:
    print("No file selected.")

Dataset loaded successfully!
Shape: (15, 6)


,rank,player,team,matches,catches,cpm
0,1,Rinku Singh,KKR,8,9,1.285
1,2,Devdutt Padikkal,RCB,8,9,1.125
2,3,Dewald Brevis,CSK,5,8,1.600
3,4,Glenn Phillips,GT,6,8,1.333
4,5,Xavier Bartlett,PBKS,7,7,1.000


### : Fielding Dataset Inspection

The fielding statistics dataset was inspected to understand its structure, data types, missing values, and duplicate records. This initial check helps assess data quality before performing further validation.

In [22]:
# Step 1: Inspect fielding statistics

print("Dataset shape:", fielding_stats.shape)

print("\nColumn names:")
print(fielding_stats.columns.tolist())

print("\nData types:")
print(fielding_stats.dtypes)

print("\nMissing values:")
print(fielding_stats.isnull().sum())

print("\nExact duplicate rows:",
      fielding_stats.duplicated().sum())

print("\nFirst 5 records:")
display(fielding_stats.head())

Dataset shape: (15, 6)

Column names:
['rank', 'player', 'team', 'matches', 'catches', 'cpm']

Data types:
rank         int64
player      object
team        object
matches      int64
catches      int64
cpm        float64
dtype: object

Missing values:
rank       0
player     0
team       0
matches    0
catches    0
cpm        0
dtype: int64

Exact duplicate rows: 0

First 5 records:


,rank,player,team,matches,catches,cpm
0,1,Rinku Singh,KKR,8,9,1.285
1,2,Devdutt Padikkal,RCB,8,9,1.125
2,3,Dewald Brevis,CSK,5,8,1.600
3,4,Glenn Phillips,GT,6,8,1.333
4,5,Xavier Bartlett,PBKS,7,7,1.000


### Step 2: Logical Consistency Checks

In this step, we check the fielding dataset for invalid or unusual values. We verify player rankings, matches played, catches, and whether the reported catches-per-match figures are consistent with the underlying data.

In [23]:
# Step 2: Logical consistency checks

# 1. Check negative numeric values
numeric_cols = fielding_stats.select_dtypes(include="number").columns

print("Negative values by column:")
print((fielding_stats[numeric_cols] < 0).sum())

# 2. Check duplicate player names
print("\nDuplicate player names:",
      fielding_stats["player"].duplicated().sum())

# 3. Check duplicate ranks
print("Duplicate ranks:",
      fielding_stats["rank"].duplicated().sum())

# 4. Check zero matches with nonzero catches
print("Zero matches with nonzero catches:",
      ((fielding_stats["matches"] == 0) &
       (fielding_stats["catches"] > 0)).sum())

# 5. Check catches per match
fielding_stats["recalculated_cpm"] = (
    fielding_stats["catches"] / fielding_stats["matches"]
).round(3)

fielding_stats["cpm_difference"] = (
    fielding_stats["recalculated_cpm"] - fielding_stats["cpm"]
).abs()

cpm_mismatches = fielding_stats[
    fielding_stats["cpm_difference"] > 0.01
]

print("\nCatches-per-match mismatches:", len(cpm_mismatches))

display(
    cpm_mismatches[
        ["player", "matches", "catches", "cpm", "recalculated_cpm"]
    ]
)

Negative values by column:
rank       0
matches    0
catches    0
cpm        0
dtype: int64

Duplicate player names: 0
Duplicate ranks: 0
Zero matches with nonzero catches: 0

Catches-per-match mismatches: 1


,player,matches,catches,cpm,recalculated_cpm
0,Rinku Singh,8,9,1.285,1.125


## : Inspect the mismatch

In [24]:
# Inspect catches-per-match mismatch

display(
    cpm_mismatches[
        ["rank", "player", "team", "matches",
         "catches", "cpm", "recalculated_cpm",
         "cpm_difference"]
    ]
)

,rank,player,team,matches,catches,cpm,recalculated_cpm,cpm_difference
0,1,Rinku Singh,KKR,8,9,1.285,1.125,0.16


#### Finding: Catches-per-Match Inconsistency

The validation identified one mismatch in the catches-per-match (`cpm`) metric.

- **Player:** Rinku Singh (KKR)
- **Matches:** 8
- **Catches:** 9
- **Reported CPM:** 1.285
- **Recalculated CPM:** 1.125

The reported value does not match the recalculated value. The discrepancy is too large to be explained by rounding alone.

**Decision:** Flag this record for source verification. Retain the original value until its accuracy can be confirmed.

### Step 3: Statistical Summary

Descriptive statistics are used to understand the distribution of matches played, catches, and catches per match among the selected players.

In [25]:
# Step 3: Statistical summary

summary_cols = ["matches", "catches", "cpm"]

display(
    fielding_stats[summary_cols]
    .describe()
    .round(3)
    .T
)

,count,mean,std,min,25%,50%,75%,max
matches,15.0,7.133,0.915,5.000,7.000,7.000,8.000,8.0
catches,15.0,6.600,1.454,5.000,5.000,7.000,7.500,9.0
cpm,15.0,0.956,0.282,0.625,0.714,0.875,1.062,1.6


## Statistical Summary: 
The fielding dataset contains 15 player records, with an average of 7.13 matches and 6.60 catches per player. The reported catches-per-match rate averages approximately 0.956. CPM values range from 0.625 to 1.600. One CPM inconsistency was identified for Rinku Singh, so the CPM statistics are provisional. No source values were changed during the audit.

## Step 4: Final Fielding Data Audit Conclusion

### Audit Results

The fielding dataset contains 15 player records and 6 original columns.

- **Missing values:** No missing values were found.
- **Duplicate records:** No exact duplicate rows were found.
- **Player and rank uniqueness:** No duplicate player names or ranks were identified.
- **Negative values:** No negative values were found in the checked numeric columns.
- **Logical checks:** No players had zero matches with nonzero catches.
- **CPM validation:** One inconsistency was identified in Rinku Singh's record.

### Issue Identified

Rinku Singh has 8 matches and 9 catches. His reported catches per match (CPM) is 1.285, whereas the recalculated value is 1.125.

This discrepancy was flagged for verification. The original value was retained because no independent source was used to confirm the correct figure.

### Final Conclusion

The fielding dataset passed the majority of structural and logical validation checks. One inconsistency was found in the catches-per-match metric. The dataset is suitable for further exploratory analysis, provided this issue is documented and the CPM figures are treated cautiously.

No source data was modified during the audit.

In [26]:
# Step 4: Final audit status

audit_summary = pd.DataFrame({
    "Audit Check": [
        "Missing values",
        "Exact duplicate rows",
        "Duplicate player names",
        "Duplicate ranks",
        "Negative numeric values",
        "Zero matches with nonzero catches",
        "CPM mismatches"
    ],
    "Result": [
        "Passed",
        "Passed",
        "Passed",
        "Passed",
        "Passed",
        "Passed",
        "1 issue flagged"
    ],
    "Status": [
        "Pass", "Pass", "Pass", "Pass", "Pass", "Pass", "Review"
    ]
})

display(audit_summary)

,Audit Check,Result,Status
0,Missing values,Passed,Pass
1,Exact duplicate rows,Passed,Pass
2,Duplicate player names,Passed,Pass
3,Duplicate ranks,Passed,Pass
4,Negative numeric values,Passed,Pass
5,Zero matches with nonzero catches,Passed,Pass
6,CPM mismatches,1 issue flagged,Review


## Points Table Dataset Audit

### Step 1: Dataset Overview

The first step is to inspect the structure of the points table, including its dimensions, column names, data types, sample records, missing values, and duplicate rows.

In [27]:
import pandas as pd
from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select Points Table CSV",
    filetypes=[("CSV files", "*.csv")]
)

points_table = pd.read_csv(file_path)

print("Dataset shape:", points_table.shape)
print("\nColumn names:")
print(points_table.columns.tolist())

print("\nData types:")
print(points_table.dtypes)

print("\nFirst 5 rows:")
display(points_table.head())

print("\nMissing values:")
display(points_table.isnull().sum())

print("\nExact duplicate rows:", points_table.duplicated().sum())

Dataset shape: (10, 9)

Column names:
['position', 'team', 'matches', 'wins', 'defeats', 'ties', 'abandoned', 'points', 'nrr']

Data types:
position       int64
team          object
matches        int64
wins           int64
defeats        int64
ties           int64
abandoned      int64
points         int64
nrr          float64
dtype: object

First 5 rows:


,position,team,matches,wins,defeats,ties,abandoned,points,nrr
0,1,Punjab Kings,7,6,0,0,1,13,1.333
1,2,Royal Challengers Bengaluru,8,6,2,0,0,12,1.919
2,3,Sunrisers Hyderabad,8,5,3,0,0,10,0.815
3,4,Rajasthan Royals,8,5,3,0,0,10,0.602
4,5,Gujarat Titans,8,4,4,0,0,8,-0.475



Missing values:


position     0
team         0
matches      0
wins         0
defeats      0
ties         0
abandoned    0
points       0
nrr          0
dtype: int64


Exact duplicate rows: 0


### Step 2: Logical Validation

In this step, we validate the points table by checking:
- Negative values in numeric columns
- Duplicate team names and positions
- Whether wins, defeats, ties, and abandoned matches add up to total matches
- Whether points match the expected points calculation

In [28]:
# Step 2: Logical validation checks

# 1. Check negative values
numeric_cols = [
    "position", "matches", "wins", "defeats",
    "ties", "abandoned", "points", "nrr"
]

print("Negative values:")
display((points_table[numeric_cols] < 0).sum())

# 2. Check duplicate team names and positions
print("Duplicate team names:", points_table["team"].duplicated().sum())
print("Duplicate positions:", points_table["position"].duplicated().sum())

# 3. Check whether match results add up to matches
points_table["calculated_matches"] = (
    points_table["wins"]
    + points_table["defeats"]
    + points_table["ties"]
    + points_table["abandoned"]
)

points_table["match_count_mismatch"] = (
    points_table["matches"] != points_table["calculated_matches"]
)

# 4. Check expected points (2 per win, 1 per tie or abandonment)
points_table["calculated_points"] = (
    2 * points_table["wins"]
    + points_table["ties"]
    + points_table["abandoned"]
)

points_table["points_mismatch"] = (
    points_table["points"] != points_table["calculated_points"]
)

print("\nMatch count mismatches:")
display(points_table.loc[
    points_table["match_count_mismatch"],
    ["team", "matches", "calculated_matches"]
])

print("\nPoints mismatches:")
display(points_table.loc[
    points_table["points_mismatch"],
    ["team", "wins", "ties", "abandoned", "points", "calculated_points"]
])

Negative values:


position     0
matches      0
wins         0
defeats      0
ties         0
abandoned    0
points       0
nrr          5
dtype: int64

Duplicate team names: 0
Duplicate positions: 0

Match count mismatches:


,team,matches,calculated_matches



Points mismatches:


,team,wins,ties,abandoned,points,calculated_points


### Step 3: Statistical Summary

Descriptive statistics help us understand the distribution of matches played, wins, defeats, points, and net run rate (NRR) across the teams in the points table.

In [29]:
# Step 3: Statistical summary

summary_cols = [
    "matches", "wins", "defeats",
    "ties", "abandoned", "points", "nrr"
]

display(points_table[summary_cols].describe().round(3).T)

,count,mean,std,min,25%,50%,75%,max
matches,10.0,7.800,0.422,7.000,8.000,8.00,8.000,8.000
wins,10.0,3.800,1.619,2.000,2.250,3.50,5.000,6.000
defeats,10.0,3.800,1.814,0.000,3.000,4.50,5.000,6.000
ties,10.0,0.000,0.000,0.000,0.000,0.00,0.000,0.000
abandoned,10.0,0.200,0.422,0.000,0.000,0.00,0.000,1.000
points,10.0,7.800,3.293,4.000,5.250,7.00,10.000,13.000
nrr,10.0,0.192,1.044,-1.106,-0.671,0.24,0.799,1.919


## Step 4: Final Points Table Audit Conclusion

### Audit Results

The points table contains 10 teams and 9 original columns.

- Missing values: None found.
- Exact duplicate rows: To be confirmed from the Step 1 output.
- Duplicate team names: None found.
- Duplicate positions: None found.
- Negative values: No negative values in the integer columns. Five teams have negative NRR values, which are valid.
- Match consistency: All teams passed the match-count validation.
- Points consistency: All teams passed the points calculation check.

### Statistical Findings

The teams have played an average of 7.8 matches and earned an average of 7.8 points. The average NRR is 0.192, with values ranging from -1.106 to 1.919.

### Final Conclusion

The points table passed the logical validation checks performed. No inconsistencies were found in the match counts or points calculations under the assumed points system.

The dataset is suitable for further analysis, subject to verification against an authoritative source if official standings accuracy is required.

No original source values were modified during the audit.

## Squads Dataset Audit

### Step 1: Dataset Overview

The first step is to inspect the squads dataset to understand its structure, column names, data types, missing values, and duplicate records.

In [30]:
import pandas as pd
from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select Squads CSV",
    filetypes=[("CSV files", "*.csv")]
)

squads = pd.read_csv(file_path)

print("Dataset shape:", squads.shape)

print("\nColumn names:")
print(squads.columns.tolist())

print("\nData types:")
print(squads.dtypes)

print("\nFirst 5 rows:")
display(squads.head())

print("\nMissing values:")
display(squads.isnull().sum())

print("\nExact duplicate rows:", squads.duplicated().sum())

Dataset shape: (257, 6)

Column names:
['team_no', 'team_name', 'player', 'nationality', 'role', 'designation']

Data types:
team_no         int64
team_name      object
player         object
nationality    object
role           object
designation    object
dtype: object

First 5 rows:


,team_no,team_name,player,nationality,role,designation
0,1,Mumbai Indians,Hardik Pandya,Indian,All Rounder,Captain
1,1,Mumbai Indians,Rohit Sharma,Indian,Batter,NaN
2,1,Mumbai Indians,Suryakumar Yadav,Indian,Batter,NaN
3,1,Mumbai Indians,Jasprit Bumrah,Indian,Bowler,NaN
4,1,Mumbai Indians,Ryan Rickelton,Overseas,Wicket Keeper,NaN



Missing values:


team_no          0
team_name        0
player           0
nationality      0
role             0
designation    247
dtype: int64


Exact duplicate rows: 0


### Step 2: Logical Validation

In this step, we validate the squads dataset by checking:
- Missing values in each column
- Duplicate player records
- Duplicate player names within teams
- Number of players assigned to each team
- Unique team numbers and team names
- Valid player roles and designations

In [31]:
# Step 2: Logical validation checks

# 1. Missing values
print("Missing values:")
display(squads.isnull().sum())

# 2. Duplicate records
print("\nExact duplicate rows:", squads.duplicated().sum())

print(
    "Duplicate player-team combinations:",
    squads.duplicated(subset=["team_name", "player"]).sum()
)

# 3. Team number and team name consistency
print("\nUnique team numbers:", squads["team_no"].nunique())
print("Unique team names:", squads["team_name"].nunique())

team_mapping = squads.groupby("team_no")["team_name"].nunique()
print("Team numbers mapped to multiple names:", (team_mapping > 1).sum())

# 4. Players per team
print("\nPlayers per team:")
display(
    squads.groupby(["team_no", "team_name"])
    .size()
    .reset_index(name="player_count")
)

# 5. Role and designation categories
print("\nPlayer roles:")
display(squads["role"].value_counts(dropna=False))

print("\nDesignations:")
display(squads["designation"].value_counts(dropna=False))

Missing values:


team_no          0
team_name        0
player           0
nationality      0
role             0
designation    247
dtype: int64


Exact duplicate rows: 0
Duplicate player-team combinations: 0

Unique team numbers: 10
Unique team names: 10
Team numbers mapped to multiple names: 0

Players per team:


,team_no,team_name,player_count
0,1,Mumbai Indians,25
1,2,Chennai Super Kings,26
2,3,Kolkata Night Riders,28
3,4,Royal Challengers Bengaluru,25
4,5,Gujrat Titans,26
5,6,Rajasthan Royals,26
6,7,Sunrisers Hyderabad,26
7,8,Punjab Kings,25
8,9,Delhi Capitals,25
9,10,Lucknow Super Giants,25



Player roles:


role
Bowler           106
All Rounder       68
Batter            54
Wicket Keeper     29
Name: count, dtype: int64


Designations:


designation
NaN        247
Captain     10
Name: count, dtype: int64

### Step 3: Category and Designation Analysis

This step examines the distribution of player roles, nationalities, and designations. It also identifies the players who have a recorded designation so that missing values can be interpreted correctly.

In [32]:
# Step 3: Category and designation analysis

print("Player role distribution:")
display(squads["role"].value_counts(dropna=False))

print("\nNationality distribution:")
display(squads["nationality"].value_counts(dropna=False))

print("\nDesignation distribution:")
display(squads["designation"].value_counts(dropna=False))

print("\nPlayers with a recorded designation:")
display(
    squads.loc[
        squads["designation"].notna(),
        ["team_name", "player", "role", "designation"]
    ]
)

Player role distribution:


role
Bowler           106
All Rounder       68
Batter            54
Wicket Keeper     29
Name: count, dtype: int64


Nationality distribution:


nationality
Indian      175
Overseas     82
Name: count, dtype: int64


Designation distribution:


designation
NaN        247
Captain     10
Name: count, dtype: int64


Players with a recorded designation:


,team_name,player,role,designation
0,Mumbai Indians,Hardik Pandya,All Rounder,Captain
25,Chennai Super Kings,Ruturaj Gaikwad,Batter,Captain
51,Kolkata Night Riders,Ajinkya Rahane,Batter,Captain
79,Royal Challengers Bengaluru,Rajat Patidar,Batter,Captain
104,Gujrat Titans,Shubman Gill,Batter,Captain
130,Rajasthan Royals,Riyan Parag,Batter,Captain
156,Sunrisers Hyderabad,Ishan Kishan,Wicket Keeper,Captain
182,Punjab Kings,Shreyas Iyer,Batter,Captain
207,Delhi Capitals,Axar Patel,All Rounder,Captain
232,Lucknow Super Giants,Rishabh Pant,Wicket Keeper,Captain


## Step 4: Final Squads Dataset Audit Conclusion

### Audit Results

The squads dataset contains 257 player records and 6 columns.

- Missing values: No missing values in essential fields. The designation column has 247 missing values.
- Duplicate records: No exact duplicate rows or duplicate player-team combinations were found.
- Team consistency: 10 unique team numbers and 10 unique team names were identified.
- Team sizes: Each team has between 25 and 28 player records.
- Player roles: All 257 players have a recorded role.
- Nationality: 175 players are Indian and 82 are overseas.
- Designations: 10 players have the designation Captain, while 247 have no designation.
- Naming review: The team names "Kolkata Night Riders" and "Gujrat Titans" should be checked against the intended official names.

### Final Conclusion

The squads dataset passed the duplicate and basic structural checks. The missing designations appear consistent with a column used to identify captains, although captain assignments and team names require verification.

No original source values were modified during the audit.